# Purpose
Train/evaluate deep learning models for aptamer-molecule interaction prediction and export benchmark outputs.

# Inputs
- `dataset/AptaBench_dataset.csv`
- `dataset/splits/*.json`
- Model and encoder components from `src.models` and `src.encoders`

# Outputs
- `artifacts/dl/screening/comprehensive_screening_*.csv`
- `artifacts/dl/screening/dl_leaderboard.csv`
- `artifacts/dl/ablations/ablation_study_gena_unimol.csv`

# Runtime (CPU/GPU, approximate)
- GPU recommended, CPU possible for quick checks only
- ~1-6 hours depending on quick/full mode and encoder selection

# Determinism/Seeds
- Seed controlled by shared config (`SEED`); reproducibility depends on backend determinism settings.

In [ ]:
# Reviewer reproducibility config
from pathlib import Path

SEED = 42
QUICK_RUN = True
FULL_RUN = not QUICK_RUN
DEVICE = "cpu"  # set to "cuda" for full GPU runs

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PATHS = {
    "dataset_csv": REPO_ROOT / "dataset" / "AptaBench_dataset.csv",
    "splits_dir": REPO_ROOT / "dataset" / "splits",
    "dl_screening_dir": REPO_ROOT / "artifacts" / "dl" / "screening",
    "dl_ablations_dir": REPO_ROOT / "artifacts" / "dl" / "ablations",
}
PATHS["dl_screening_dir"].mkdir(parents=True, exist_ok=True)
PATHS["dl_ablations_dir"].mkdir(parents=True, exist_ok=True)
print({"SEED": SEED, "QUICK_RUN": QUICK_RUN, "DEVICE": DEVICE})

In [ ]:
import os
import json
import warnings
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    matthews_corrcoef, mean_squared_error, mean_absolute_error, r2_score
)

# Optuna
import optuna
from optuna.samplers import TPESampler

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns

# Model visualization and parameter counting
try:
    from torchinfo import summary
    TORCHINFO_AVAILABLE = True
except ImportError:
    TORCHINFO_AVAILABLE = False
    print("⚠ torchinfo not available - will use manual parameter counting")

# Set random seeds
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


### 1.2. Reuse Existing Project Functions (with Fallbacks)


In [ ]:
# Try to import existing functions, create fallbacks if needed

# === Data Loading ===
try:
    from src.models.screening import load_splits
    print("✓ Using existing load_splits")
except ImportError:
    def load_splits(split_mode, base_dir="../dataset/splits"):
        """Fallback: Load precomputed splits from JSON."""
        path = os.path.join(base_dir, f"{split_mode}.json")
        with open(path, "r") as f:
            data = json.load(f)
        return [(np.array(d["train_idx"]), np.array(d["val_idx"])) for d in data]
    print("⚠ Created fallback load_splits")

# === Encoders ===
try:
    from src.encoders.aptamer_encoders import (
        onehot_with_type_bit, kmer_freq_with_type_bit,
        gena_embed, dnabert2_embed
    )
    print("✓ Using existing aptamer encoders")
except ImportError:
    print("⚠ Aptamer encoders not found - will need to implement fallbacks")

try:
    from src.encoders.molecule_encoders import (
        morgan_fp, maccs_fp, physchem_descriptors,
        chemberta_embed, unimol_embed, molformer_embed
    )
    print("✓ Using existing molecule encoders")
except ImportError:
    print("⚠ Molecule encoders not found - will need to implement fallbacks")

# === Load Precomputed Embeddings ===
try:
    from src.models.screening import load_embeddings
    print("✓ Using existing load_embeddings function")
except ImportError:
    def load_embeddings(df, apt_cfgs, mol_cfgs, embeddings_dir="../artifacts/cache/embeddings"):
        """Fallback: Load precomputed embeddings from cache."""
        import hashlib
        from pathlib import Path
        
        # Compute dataset hash
        data_str = f"{df.shape}_{df.iloc[:5].to_json()}_{list(df.columns)}"
        dataset_hash = hashlib.md5(data_str.encode()).hexdigest()
        
        cache_dir = Path(embeddings_dir)
        if not cache_dir.exists():
            return None
        
        apt_map = {}
        mol_map = {}
        
        # Load aptamer embeddings
        for cfg in apt_cfgs:
            name = cfg["name"]
            cache_file = cache_dir / f"apt_{name}_{dataset_hash}.npy"
            if cache_file.exists():
                apt_map[name] = np.load(cache_file)
            else:
                return None
        
        # Load molecule embeddings
        for cfg in mol_cfgs:
            name = cfg["name"]
            cache_file = cache_dir / f"mol_{name}_{dataset_hash}.npy"
            if cache_file.exists():
                mol_map[name] = np.load(cache_file)
            else:
                return None
        
        return apt_map, mol_map
    print("⚠ Created fallback load_embeddings")


### 1.4. Model Visualization and Parameter Counting Utilities


In [ ]:
def count_parameters(model):
    """Count total and trainable parameters in a model."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'total': total_params,
        'trainable': trainable_params,
        'non_trainable': total_params - trainable_params
    }


def format_parameter_count(count):
    """Format parameter count in human-readable format."""
    if count >= 1e9:
        return f"{count / 1e9:.2f}B"
    elif count >= 1e6:
        return f"{count / 1e6:.2f}M"
    elif count >= 1e3:
        return f"{count / 1e3:.2f}K"
    else:
        return str(count)


def visualize_model_architecture(model, model_name, apt_dim, mol_dim, save_path=None):
    """
    Visualize model architecture as a diagram.
    """
    from matplotlib.patches import FancyArrowPatch
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis('off')
    
    # Title
    ax.text(5, 11.5, f"{model_name} Architecture", 
            ha='center', va='top', fontsize=16, fontweight='bold')
    
    # Count parameters
    params = count_parameters(model)
    param_text = f"Total: {format_parameter_count(params['total'])} | "
    param_text += f"Trainable: {format_parameter_count(params['trainable'])}"
    ax.text(5, 11, param_text, ha='center', va='top', fontsize=10, style='italic')
    
    # Draw architecture based on model type
    if 'PretrainedEncoder' in model_name:
        # Aptamer branch
        apt_box = FancyBboxPatch((0.5, 8), 2, 1.5, boxstyle="round,pad=0.1", 
                                facecolor='lightblue', edgecolor='black', linewidth=2)
        ax.add_patch(apt_box)
        ax.text(1.75, 9, f"Aptamer\nInput\n({apt_dim})", ha='center', va='center', fontsize=9)
        
        # Molecule branch
        mol_box = FancyBboxPatch((7.5, 8), 2, 1.5, boxstyle="round,pad=0.1",
                                facecolor='lightgreen', edgecolor='black', linewidth=2)
        ax.add_patch(mol_box)
        ax.text(8.75, 9, f"Molecule\nInput\n({mol_dim})", ha='center', va='center', fontsize=9)
        
        # Projection layers
        apt_proj = FancyBboxPatch((0.5, 6), 2, 1, boxstyle="round,pad=0.1",
                                 facecolor='lightblue', edgecolor='black', linewidth=1.5)
        ax.add_patch(apt_proj)
        ax.text(1.75, 6.5, "Projection\nLayers", ha='center', va='center', fontsize=8)
        
        mol_proj = FancyBboxPatch((7.5, 6), 2, 1, boxstyle="round,pad=0.1",
                                 facecolor='lightgreen', edgecolor='black', linewidth=1.5)
        ax.add_patch(mol_proj)
        ax.text(8.75, 6.5, "Projection\nLayers", ha='center', va='center', fontsize=8)
        
        # Fusion
        fusion_box = FancyBboxPatch((3.5, 4), 3, 1.5, boxstyle="round,pad=0.1",
                                   facecolor='lightyellow', edgecolor='black', linewidth=2)
        ax.add_patch(fusion_box)
        ax.text(5, 4.75, "Fusion Head\n(Concatenate + MLP)", ha='center', va='center', fontsize=9)
        
        # Output
        out_box = FancyBboxPatch((4.5, 2), 1, 0.8, boxstyle="round,pad=0.1",
                                facecolor='lightcoral', edgecolor='black', linewidth=2)
        ax.add_patch(out_box)
        ax.text(5, 2.4, "Output\n(Logits)", ha='center', va='center', fontsize=9)
        
        # Arrows
        arrow1 = FancyArrowPatch((1.75, 8), (1.75, 7.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow1)
        arrow2 = FancyArrowPatch((8.75, 8), (8.75, 7.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow2)
        arrow3 = FancyArrowPatch((1.75, 6), (3.5, 5.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow3)
        arrow4 = FancyArrowPatch((8.75, 6), (6.5, 5.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow4)
        arrow5 = FancyArrowPatch((5, 4), (5, 3.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow5)
        
    elif 'TwoTower' in model_name:
        # Two towers
        apt_tower = FancyBboxPatch((0.5, 7), 2.5, 3, boxstyle="round,pad=0.1",
                                   facecolor='lightblue', edgecolor='black', linewidth=2)
        ax.add_patch(apt_tower)
        ax.text(1.75, 8.5, "Aptamer Tower\n(MLP Layers)", ha='center', va='center', fontsize=9)
        
        mol_tower = FancyBboxPatch((7, 7), 2.5, 3, boxstyle="round,pad=0.1",
                                  facecolor='lightgreen', edgecolor='black', linewidth=2)
        ax.add_patch(mol_tower)
        ax.text(8.25, 8.5, "Molecule Tower\n(MLP Layers)", ha='center', va='center', fontsize=9)
        
        # Fusion
        fusion_box = FancyBboxPatch((3.5, 4), 3, 1.5, boxstyle="round,pad=0.1",
                                   facecolor='lightyellow', edgecolor='black', linewidth=2)
        ax.add_patch(fusion_box)
        ax.text(5, 4.75, "Fusion Layers\n(MLP)", ha='center', va='center', fontsize=9)
        
        # Output
        out_box = FancyBboxPatch((4.5, 2), 1, 0.8, boxstyle="round,pad=0.1",
                                facecolor='lightcoral', edgecolor='black', linewidth=2)
        ax.add_patch(out_box)
        ax.text(5, 2.4, "Output", ha='center', va='center', fontsize=9)
        
        # Arrows
        arrow1 = FancyArrowPatch((1.75, 7), (1.75, 6.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow1)
        arrow2 = FancyArrowPatch((8.25, 7), (8.25, 6.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow2)
        arrow3 = FancyArrowPatch((1.75, 6.5), (3.5, 5.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow3)
        arrow4 = FancyArrowPatch((8.25, 6.5), (6.5, 5.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow4)
        arrow5 = FancyArrowPatch((5, 4), (5, 3.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow5)
        
    elif 'CrossAttention' in model_name:
        # Inputs
        apt_in = FancyBboxPatch((0.5, 8.5), 2, 1, boxstyle="round,pad=0.1",
                               facecolor='lightblue', edgecolor='black', linewidth=2)
        ax.add_patch(apt_in)
        ax.text(1.75, 9, f"Aptamer\n({apt_dim})", ha='center', va='center', fontsize=9)
        
        mol_in = FancyBboxPatch((7.5, 8.5), 2, 1, boxstyle="round,pad=0.1",
                               facecolor='lightgreen', edgecolor='black', linewidth=2)
        ax.add_patch(mol_in)
        ax.text(8.75, 9, f"Molecule\n({mol_dim})", ha='center', va='center', fontsize=9)
        
        # Projections
        apt_proj = FancyBboxPatch((0.5, 7), 2, 0.8, boxstyle="round,pad=0.1",
                                 facecolor='lightblue', edgecolor='black', linewidth=1.5)
        ax.add_patch(apt_proj)
        ax.text(1.75, 7.4, "Project", ha='center', va='center', fontsize=8)
        
        mol_proj = FancyBboxPatch((7.5, 7), 2, 0.8, boxstyle="round,pad=0.1",
                                 facecolor='lightgreen', edgecolor='black', linewidth=1.5)
        ax.add_patch(mol_proj)
        ax.text(8.75, 7.4, "Project", ha='center', va='center', fontsize=8)
        
        # Cross-attention
        attn_box = FancyBboxPatch((3, 5), 4, 1.5, boxstyle="round,pad=0.1",
                                 facecolor='lightyellow', edgecolor='black', linewidth=2)
        ax.add_patch(attn_box)
        ax.text(5, 5.75, "Cross-Attention\nLayers", ha='center', va='center', fontsize=9)
        
        # Classifier
        cls_box = FancyBboxPatch((3.5, 3), 3, 1, boxstyle="round,pad=0.1",
                                facecolor='lightcoral', edgecolor='black', linewidth=2)
        ax.add_patch(cls_box)
        ax.text(5, 3.5, "Classifier Head", ha='center', va='center', fontsize=9)
        
        # Output
        out_box = FancyBboxPatch((4.5, 1.5), 1, 0.8, boxstyle="round,pad=0.1",
                                facecolor='lightcoral', edgecolor='black', linewidth=2)
        ax.add_patch(out_box)
        ax.text(5, 1.9, "Output", ha='center', va='center', fontsize=9)
        
        # Arrows
        arrow1 = FancyArrowPatch((1.75, 8.5), (1.75, 8.2), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow1)
        arrow2 = FancyArrowPatch((8.75, 8.5), (8.75, 8.2), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow2)
        arrow3 = FancyArrowPatch((1.75, 7), (3, 6.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow3)
        arrow4 = FancyArrowPatch((8.75, 7), (7, 6.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow4)
        arrow5 = FancyArrowPatch((5, 5), (5, 4.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow5)
        arrow6 = FancyArrowPatch((5, 3), (5, 2.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow6)
        
    elif 'MultiTask' in model_name:
        # Inputs
        input_box = FancyBboxPatch((3, 9), 4, 1, boxstyle="round,pad=0.1",
                                   facecolor='lightblue', edgecolor='black', linewidth=2)
        ax.add_patch(input_box)
        ax.text(5, 9.5, f"Concatenated Input\n({apt_dim + mol_dim})", ha='center', va='center', fontsize=9)
        
        # Shared encoder
        shared_box = FancyBboxPatch((2.5, 7), 5, 1.5, boxstyle="round,pad=0.1",
                                    facecolor='lightyellow', edgecolor='black', linewidth=2)
        ax.add_patch(shared_box)
        ax.text(5, 7.75, "Shared Encoder\n(MLP Layers)", ha='center', va='center', fontsize=9)
        
        # Task heads
        cls_head = FancyBboxPatch((1, 4.5), 2.5, 1.5, boxstyle="round,pad=0.1",
                                 facecolor='lightcoral', edgecolor='black', linewidth=2)
        ax.add_patch(cls_head)
        ax.text(2.25, 5.25, "Classification\nHead", ha='center', va='center', fontsize=9)
        
        reg_head = FancyBboxPatch((6.5, 4.5), 2.5, 1.5, boxstyle="round,pad=0.1",
                                 facecolor='lightgreen', edgecolor='black', linewidth=2)
        ax.add_patch(reg_head)
        ax.text(7.75, 5.25, "Regression\nHead", ha='center', va='center', fontsize=9)
        
        # Outputs
        cls_out = FancyBboxPatch((1.5, 2.5), 1.5, 0.8, boxstyle="round,pad=0.1",
                                facecolor='lightcoral', edgecolor='black', linewidth=2)
        ax.add_patch(cls_out)
        ax.text(2.25, 2.9, "Cls\nOutput", ha='center', va='center', fontsize=8)
        
        reg_out = FancyBboxPatch((7, 2.5), 1.5, 0.8, boxstyle="round,pad=0.1",
                                facecolor='lightgreen', edgecolor='black', linewidth=2)
        ax.add_patch(reg_out)
        ax.text(7.75, 2.9, "Reg\nOutput", ha='center', va='center', fontsize=8)
        
        # Arrows
        arrow1 = FancyArrowPatch((5, 9), (5, 8.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow1)
        arrow2 = FancyArrowPatch((5, 7), (2.25, 6.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow2)
        arrow3 = FancyArrowPatch((5, 7), (7.75, 6.5), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow3)
        arrow4 = FancyArrowPatch((2.25, 4.5), (2.25, 4), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow4)
        arrow5 = FancyArrowPatch((7.75, 4.5), (7.75, 4), arrowstyle='->', mutation_scale=20, linewidth=1.5, color='black')
        ax.add_patch(arrow5)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Architecture saved to: {save_path}")
    plt.show()
    
    return params


print("✓ Visualization utilities ready")


### 1.3. Metrics Utilities (with Fallbacks)


In [ ]:
def compute_classification_metrics(y_true, y_pred, y_scores=None):
    """
    Compute classification metrics: MCC, ROC-AUC, PR-AUC, F1.
    
    Args:
        y_true: Ground truth binary labels
        y_pred: Binary predictions
        y_scores: Continuous scores (probabilities) for ROC/PR-AUC
    
    Returns:
        dict with metrics
    """
    metrics = {}
    
    # MCC
    metrics['MCC'] = matthews_corrcoef(y_true, y_pred)
    
    # F1
    metrics['F1'] = f1_score(y_true, y_pred, zero_division=0)
    
    # ROC-AUC and PR-AUC (require continuous scores)
    if y_scores is not None and len(np.unique(y_true)) > 1:
        metrics['ROC-AUC'] = roc_auc_score(y_true, y_scores)
        metrics['PR-AUC'] = average_precision_score(y_true, y_scores)
    else:
        metrics['ROC-AUC'] = np.nan
        metrics['PR-AUC'] = np.nan
    
    return metrics


def compute_regression_metrics(y_true, y_pred):
    """
    Compute regression metrics: RMSE, MAE, R².
    
    Args:
        y_true: Ground truth values
        y_pred: Predictions
    
    Returns:
        dict with metrics
    """
    metrics = {}
    metrics['RMSE'] = np.sqrt(mean_squared_error(y_true, y_pred))
    metrics['MAE'] = mean_absolute_error(y_true, y_pred)
    metrics['R2'] = r2_score(y_true, y_pred)
    return metrics


def aggregate_metrics(metrics_list):
    """
    Aggregate metrics across folds: mean ± std.
    
    Args:
        metrics_list: List of metric dicts (one per fold)
    
    Returns:
        dict with aggregated metrics
    """
    if not metrics_list:
        return {}
    
    # Collect all metric names
    all_keys = set()
    for m in metrics_list:
        all_keys.update(m.keys())
    
    aggregated = {}
    for key in all_keys:
        values = [m.get(key, np.nan) for m in metrics_list]
        values = np.array(values)
        values = values[~np.isnan(values)]  # Remove NaN
        
        if len(values) > 0:
            aggregated[f"{key} mean"] = np.mean(values)
            aggregated[f"{key} std"] = np.std(values)
        else:
            aggregated[f"{key} mean"] = np.nan
            aggregated[f"{key} std"] = np.nan
    
    return aggregated


print("✓ Metrics utilities ready")


## 2. Data Loading and Preparation


In [ ]:
# Paths
DATASET_PATH = "../dataset/AptaBench_dataset.csv"
SPLITS_DIR = "../dataset/splits"
ARTIFACTS_DIR = "../artifacts/dl"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "optuna"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "metrics"), exist_ok=True)

# Load dataset
df = pd.read_csv(DATASET_PATH)
print(f"Dataset loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\npKd statistics:\n{df['pKd_value'].describe()}")

# Prepare data
df = df.reset_index(drop=True)
y_class = df['label'].astype(int).values
y_reg = df['pKd_value'].values  # For regression task

# Check for missing pKd values
has_pkd = ~np.isnan(y_reg)
print(f"\nSamples with pKd: {has_pkd.sum()} / {len(df)}")


## 3. PyTorch Dataset and DataLoader


In [ ]:
class AptamerMoleculeDataset(Dataset):
    """PyTorch Dataset for aptamer-molecule pairs."""
    
    def __init__(self, apt_features, mol_features, labels_class, labels_reg=None):
        """
        Args:
            apt_features: np.ndarray, aptamer features [N, D_apt]
            mol_features: np.ndarray, molecule features [N, D_mol]
            labels_class: np.ndarray, binary classification labels [N]
            labels_reg: np.ndarray, regression targets [N] (optional)
        """
        self.apt_features = torch.FloatTensor(apt_features)
        self.mol_features = torch.FloatTensor(mol_features)
        self.labels_class = torch.LongTensor(labels_class)
        self.labels_reg = torch.FloatTensor(labels_reg) if labels_reg is not None else None
    
    def __len__(self):
        return len(self.labels_class)
    
    def __getitem__(self, idx):
        item = {
            'apt': self.apt_features[idx],
            'mol': self.mol_features[idx],
            'label_class': self.labels_class[idx]
        }
        if self.labels_reg is not None:
            item['label_reg'] = self.labels_reg[idx]
        return item


print("✓ Dataset class defined")


In [ ]:
"""
Deep learning encoders with pretrained models and various top architectures.
"""

import math

class IdentityTop(nn.Module):
    """Identity top layer with optional linear projection."""
    
    def __init__(self, input_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, attention_mask=None) -> torch.Tensor:
        """Forward pass. x: (batch_size, seq_len, input_dim) -> (batch_size, output_dim)"""
        # Apply linear projection
        x = self.linear(x)
        
        # Mean pooling over sequence dimension
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            x = (x * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)
        
        # Apply layer norm and dropout
        x = self.layer_norm(x)
        x = self.dropout(x)
        return x


class CNNTop(nn.Module):
    """CNN top layer with global max pooling."""
    
    def __init__(self, input_dim: int, output_dim: int, kernel_sizes=(3, 5, 7), dropout: float = 0.1):
        super().__init__()
        # Distribute output_dim across convolutions, handling remainder
        n_convs = len(kernel_sizes)
        base_channels = output_dim // n_convs
        remainder = output_dim % n_convs
        
        # Create convolutions with remainder distributed to first few layers
        self.convs = nn.ModuleList()
        for i, k in enumerate(kernel_sizes):
            # Add 1 extra channel to first 'remainder' convolutions
            out_channels = base_channels + (1 if i < remainder else 0)
            self.convs.append(nn.Conv1d(input_dim, out_channels, k, padding=k//2))
        
        # Verify total output dimension matches output_dim
        actual_output_dim = sum(conv.out_channels for conv in self.convs)
        assert actual_output_dim == output_dim, f"Expected output_dim={output_dim}, got {actual_output_dim}"
        
        self.layer_norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, attention_mask=None) -> torch.Tensor:
        """Forward pass. x: (batch_size, seq_len, input_dim) -> (batch_size, output_dim)"""
        # Transpose for Conv1d: (batch_size, input_dim, seq_len)
        x = x.transpose(1, 2)
        
        # Apply convolutions
        conv_outputs = []
        for conv in self.convs:
            conv_out = F.relu(conv(x))
            # Global max pooling
            conv_out = F.adaptive_max_pool1d(conv_out, 1).squeeze(-1)
            conv_outputs.append(conv_out)
        
        # Concatenate outputs
        x = torch.cat(conv_outputs, dim=1)
        
        # Apply layer norm and dropout
        x = self.layer_norm(x)
        x = self.dropout(x)
        return x


class AttentionPooling(nn.Module):
    """Attention-based pooling mechanism."""
    
    def __init__(self, input_dim: int):
        super().__init__()
        self.attention = nn.Linear(input_dim, 1)
        
    def forward(self, x: torch.Tensor, attention_mask=None) -> torch.Tensor:
        """Forward pass. x: (batch_size, seq_len, input_dim) -> (batch_size, input_dim)"""
        # Compute attention weights
        attention_weights = self.attention(x).squeeze(-1)  # (batch_size, seq_len)
        
        # Apply mask if provided
        if attention_mask is not None:
            mask_value = -1e4 if attention_weights.dtype == torch.float16 else -1e9
            attention_weights = attention_weights.masked_fill(attention_mask == 0, mask_value)
        
        # Apply softmax
        attention_weights = F.softmax(attention_weights, dim=-1)
        
        # Weighted sum
        x = torch.bmm(attention_weights.unsqueeze(1), x).squeeze(1)
        return x


class LSTMTop(nn.Module):
    """LSTM top layer with attention pooling."""
    
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128, 
                 num_layers: int = 2, dropout: float = 0.1, bidirectional: bool = True):
        super().__init__()
        # Use input_dim as LSTM input, hidden_dim as LSTM hidden size
        lstm_hidden = hidden_dim
        self.lstm = nn.LSTM(
            input_dim, lstm_hidden, num_layers, 
            batch_first=True, dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        lstm_output_dim = lstm_hidden * 2 if bidirectional else lstm_hidden
        self.attention = AttentionPooling(lstm_output_dim)
        self.linear = nn.Linear(lstm_output_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, attention_mask=None) -> torch.Tensor:
        """Forward pass. x: (batch_size, seq_len, input_dim) -> (batch_size, output_dim)"""
        # Apply LSTM
        lstm_out, _ = self.lstm(x)
        
        # Apply attention pooling
        x = self.attention(lstm_out, attention_mask)
        
        # Apply linear projection
        x = self.linear(x)
        
        # Apply layer norm and dropout
        x = self.layer_norm(x)
        x = self.dropout(x)
        return x


class TransformerTop(nn.Module):
    """Transformer top layer with 1-2 encoder blocks."""
    
    def __init__(self, input_dim: int, output_dim: int, num_layers: int = 2, 
                 num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=input_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.attention = AttentionPooling(input_dim)
        self.linear = nn.Linear(input_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor, attention_mask=None) -> torch.Tensor:
        """Forward pass. x: (batch_size, seq_len, input_dim) -> (batch_size, output_dim)"""
        # Create padding mask for transformer
        if attention_mask is not None:
            padding_mask = attention_mask == 0
        else:
            padding_mask = None
        
        # Apply transformer
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        
        # Apply attention pooling
        x = self.attention(x, attention_mask)
        
        # Apply linear projection
        x = self.linear(x)
        
        # Apply layer norm and dropout
        x = self.layer_norm(x)
        x = self.dropout(x)
        return x


print("✓ Top layer architectures (IdentityTop, CNNTop, LSTMTop, TransformerTop) defined")


## 4. Deep Learning Model Architectures

### 4.1. Scenario 1: Pretrained Encoders + Unfreezing Layers


In [ ]:
class PretrainedEncoderModel(nn.Module):
    """
    Scenario 1: Pretrained encoders (HF transformers) with unfreezing capability.
    Uses frozen base encoders + trainable projection layers + fusion head.
    """
    
    def __init__(
        self,
        apt_dim,
        mol_dim,
        hidden_dims=[512, 256, 128, 64],
        dropout=0.3,
        unfreeze_apt_layers=0,
        unfreeze_mol_layers=0,
        apt_encoder=None,
        mol_encoder=None
    ):
        super().__init__()
        self.apt_dim = apt_dim
        self.mol_dim = mol_dim
        self.apt_encoder = apt_encoder  
        self.mol_encoder = mol_encoder  
        
        # Deep projection layers for aptamer
        apt_layers = []
        prev_dim = apt_dim
        for dim in hidden_dims:
            apt_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        self.apt_proj = nn.Sequential(*apt_layers)
        
        # Deep projection layers for molecule
        mol_layers = []
        prev_dim = mol_dim
        for dim in hidden_dims:
            mol_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        self.mol_proj = nn.Sequential(*mol_layers)
        
        # Deep fusion head
        fusion_dim = hidden_dims[-1] * 2
        fusion_dims = [fusion_dim] + hidden_dims[:-1] + [hidden_dims[-1] // 2]
        fusion_layers = []
        for i in range(len(fusion_dims) - 1):
            fusion_layers.extend([
                nn.Linear(fusion_dims[i], fusion_dims[i+1]),
                nn.BatchNorm1d(fusion_dims[i+1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
        fusion_layers.append(nn.Linear(fusion_dims[-1], 1))
        self.fusion_head = nn.Sequential(*fusion_layers)
    
    def unfreeze_last_n_layers(self, encoder, n_layers):
        """Unfreeze last n layers of a transformer encoder."""
        if encoder is None:
            return
        # For HF models, iterate through encoder layers
        if hasattr(encoder, 'encoder') and hasattr(encoder.encoder, 'layer'):
            layers = encoder.encoder.layer
            total_layers = len(layers)
            for i in range(max(0, total_layers - n_layers), total_layers):
                for param in layers[i].parameters():
                    param.requires_grad = True
    
    def forward(self, apt, mol):
        # Project features through deep networks
        apt_emb = self.apt_proj(apt)
        mol_emb = self.mol_proj(mol)
        
        # Concatenate
        fused = torch.cat([apt_emb, mol_emb], dim=1)
        
        # Classification head
        logits = self.fusion_head(fused)
        return logits.squeeze(-1)


print("✓ Scenario 1 model defined")


### 4.2. Scenario 2: Feature-based Two-Tower MLP


In [ ]:
class TwoTowerMLP(nn.Module):
    """
    Scenario 2: Feature-based two-tower architecture.
    Each tower processes its modality, then fusion layer combines them.
    """
    
    def __init__(
        self,
        apt_dim,
        mol_dim,
        tower_hidden_dims=[1024, 512, 256, 128],
        fusion_hidden_dims=[512, 256, 128, 64],
        dropout=0.3
    ):
        super().__init__()
        
        # Aptamer tower
        apt_layers = []
        prev_dim = apt_dim
        for dim in tower_hidden_dims:
            apt_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        self.apt_tower = nn.Sequential(*apt_layers)
        
        # Molecule tower
        mol_layers = []
        prev_dim = mol_dim
        for dim in tower_hidden_dims:
            mol_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        self.mol_tower = nn.Sequential(*mol_layers)
        
        # Fusion layers
        fusion_dim = tower_hidden_dims[-1] * 2
        fusion_layers = []
        prev_dim = fusion_dim
        for dim in fusion_hidden_dims:
            fusion_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        fusion_layers.append(nn.Linear(prev_dim, 1))
        self.fusion = nn.Sequential(*fusion_layers)
    
    def forward(self, apt, mol):
        apt_emb = self.apt_tower(apt)
        mol_emb = self.mol_tower(mol)
        
        # Concatenate
        fused = torch.cat([apt_emb, mol_emb], dim=1)
        
        # Output
        logits = self.fusion(fused)
        return logits.squeeze(-1)


class TwoTowerWithTopHeads(nn.Module):
    """
    Two-tower architecture with sequence-based top heads (CNN, LSTM, Transformer).
    Adapts flat embeddings to sequences for top heads.
    Supports different top heads for aptamer and molecule.
    """
    
    def __init__(
        self,
        apt_dim,
        mol_dim,
        apt_top_type="cnn",  # "identity", "cnn", "lstm", "transformer"
        mol_top_type="cnn",  # "identity", "cnn", "lstm", "transformer"
        seq_len=32,  # Length of pseudo-sequence
        hidden_dim=256,  # Hidden dimension for sequence processing
        output_dim=128,  # Output dimension from top head
        fusion_hidden_dims=[512, 256, 128, 64],
        dropout=0.3,
        apt_top_kwargs=None,  # Additional kwargs for aptamer top head
        mol_top_kwargs=None   # Additional kwargs for molecule top head
    ):
        super().__init__()
        
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        
        # Project flat embeddings to sequences
        # We'll use a learnable projection to create sequences
        self.apt_seq_proj = nn.Sequential(
            nn.Linear(apt_dim, seq_len * hidden_dim),
            nn.LayerNorm(seq_len * hidden_dim)
        )
        self.mol_seq_proj = nn.Sequential(
            nn.Linear(mol_dim, seq_len * hidden_dim),
            nn.LayerNorm(seq_len * hidden_dim)
        )
        
        # Top heads
        top_classes = {
            "identity": IdentityTop,
            "cnn": CNNTop,
            "lstm": LSTMTop,
            "transformer": TransformerTop
        }
        
        if apt_top_type not in top_classes:
            raise ValueError(f"Unknown apt_top_type: {apt_top_type}. Must be one of {list(top_classes.keys())}")
        if mol_top_type not in top_classes:
            raise ValueError(f"Unknown mol_top_type: {mol_top_type}. Must be one of {list(top_classes.keys())}")
        
        # Filter kwargs to avoid conflicts with TwoTowerWithTopHeads params
        def filter_kwargs(kwargs_dict):
            if kwargs_dict is None:
                return {}
            return {k: v for k, v in kwargs_dict.items() 
                   if k not in ['seq_len', 'output_dim', 'fusion_hidden_dims', 'hidden_dim']}
        
        apt_kwargs = filter_kwargs(apt_top_kwargs)
        mol_kwargs = filter_kwargs(mol_top_kwargs)
        
        AptTopClass = top_classes[apt_top_type]
        MolTopClass = top_classes[mol_top_type]
        
        self.apt_top = AptTopClass(hidden_dim, output_dim, dropout=dropout, **apt_kwargs)
        self.mol_top = MolTopClass(hidden_dim, output_dim, dropout=dropout, **mol_kwargs)
        
        # Fusion layers
        fusion_dim = output_dim * 2
        fusion_layers = []
        prev_dim = fusion_dim
        for dim in fusion_hidden_dims:
            fusion_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        fusion_layers.append(nn.Linear(prev_dim, 1))
        self.fusion = nn.Sequential(*fusion_layers)
    
    def forward(self, apt, mol):
        # Project to sequences: (batch_size, feature_dim) -> (batch_size, seq_len, hidden_dim)
        apt_seq = self.apt_seq_proj(apt).view(-1, self.seq_len, self.hidden_dim)
        mol_seq = self.mol_seq_proj(mol).view(-1, self.seq_len, self.hidden_dim)
        
        # Apply top heads
        apt_emb = self.apt_top(apt_seq)
        mol_emb = self.mol_top(mol_seq)
        
        # Concatenate
        fused = torch.cat([apt_emb, mol_emb], dim=1)
        
        # Output
        logits = self.fusion(fused)
        return logits.squeeze(-1)


print("✓ Scenario 2 models defined (MLP and with Top Heads)")


### 4.3. Scenario 3: Cross-Attention Fusion


In [ ]:
class CrossAttentionFusion(nn.Module):
    """
    Cross-attention module for fusing aptamer and molecule representations.
    """
    
    def __init__(self, dim, n_heads=8, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        
        assert dim % n_heads == 0, "dim must be divisible by n_heads"
        
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(dim)
    
    def forward(self, query, key_value):
        """
        Args:
            query: [B, L_q, D] - e.g., aptamer features
            key_value: [B, L_kv, D] - e.g., molecule features
        """
        B, L_q, D = query.shape
        L_kv = key_value.shape[1]
        
        # Projections
        Q = self.q_proj(query).view(B, L_q, self.n_heads, self.head_dim).transpose(1, 2)  # [B, H, L_q, d]
        K = self.k_proj(key_value).view(B, L_kv, self.n_heads, self.head_dim).transpose(1, 2)  # [B, H, L_kv, d]
        V = self.v_proj(key_value).view(B, L_kv, self.n_heads, self.head_dim).transpose(1, 2)  # [B, H, L_kv, d]
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.head_dim)  # [B, H, L_q, L_kv]
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        out = torch.matmul(attn, V)  # [B, H, L_q, d]
        out = out.transpose(1, 2).contiguous().view(B, L_q, D)  # [B, L_q, D]
        out = self.out_proj(out)
        
        # Residual + layer norm
        out = self.layer_norm(query + out)
        return out


class CrossAttentionModel(nn.Module):
    """
    Scenario 3: Cross-attention fusion model.
    Uses cross-attention to let aptamer and molecule attend to each other.
    """
    
    def __init__(
        self,
        apt_dim,
        mol_dim,
        hidden_dim=512,
        n_attention_layers=4,
        n_heads=16,
        dropout=0.3,
        classifier_hidden_dims=[512, 256, 128]
    ):
        super().__init__()
        
        # Deep projection to common dimension
        apt_proj_layers = [
            nn.Linear(apt_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim)
        ]
        self.apt_proj = nn.Sequential(*apt_proj_layers)
        
        mol_proj_layers = [
            nn.Linear(mol_dim, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim)
        ]
        self.mol_proj = nn.Sequential(*mol_proj_layers)
        
        # Multiple cross-attention layers
        self.attention_layers = nn.ModuleList([
            CrossAttentionFusion(hidden_dim, n_heads, dropout)
            for _ in range(n_attention_layers)
        ])
        
        # Deep classification head
        cls_layers = []
        prev_dim = hidden_dim * 2
        for dim in classifier_hidden_dims:
            cls_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        cls_layers.append(nn.Linear(prev_dim, 1))
        self.classifier = nn.Sequential(*cls_layers)
    
    def forward(self, apt, mol):
        # Project to common dimension
        apt_emb = self.apt_proj(apt).unsqueeze(1)  # [B, 1, D]
        mol_emb = self.mol_proj(mol).unsqueeze(1)  # [B, 1, D]
        
        # Cross-attention: aptamer attends to molecule
        apt_attended = apt_emb
        for attn_layer in self.attention_layers:
            apt_attended = attn_layer(apt_attended, mol_emb)
        
        # Cross-attention: molecule attends to aptamer
        mol_attended = mol_emb
        for attn_layer in self.attention_layers:
            mol_attended = attn_layer(mol_attended, apt_emb)
        
        # Concatenate and classify
        fused = torch.cat([apt_attended.squeeze(1), mol_attended.squeeze(1)], dim=1)
        logits = self.classifier(fused)
        return logits.squeeze(-1)


print("✓ Scenario 3 model defined")


### 4.4. Scenario 4: Multi-Task (Classification + Regression)


In [ ]:
class MultiTaskModel(nn.Module):
    """
    Scenario 4: Multi-task model with classification and regression heads.
    Shared encoder + two separate heads.
    """
    
    def __init__(
        self,
        apt_dim,
        mol_dim,
        shared_hidden_dims=[1024, 512, 256, 128],
        task_hidden_dims=[256, 128, 64],
        dropout=0.3
    ):
        super().__init__()
        
        # Deep shared encoder
        shared_layers = []
        prev_dim = apt_dim + mol_dim
        for dim in shared_hidden_dims:
            shared_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        self.shared_encoder = nn.Sequential(*shared_layers)
        
        # Deep Classification head
        cls_layers = []
        prev_dim = shared_hidden_dims[-1]
        for dim in task_hidden_dims:
            cls_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        cls_layers.append(nn.Linear(prev_dim, 1))
        self.classifier = nn.Sequential(*cls_layers)
        
        # Deep Regression head
        reg_layers = []
        prev_dim = shared_hidden_dims[-1]
        for dim in task_hidden_dims:
            reg_layers.extend([
                nn.Linear(prev_dim, dim),
                nn.BatchNorm1d(dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = dim
        reg_layers.append(nn.Linear(prev_dim, 1))
        self.regressor = nn.Sequential(*reg_layers)
    
    def forward(self, apt, mol):
        # Concatenate inputs
        x = torch.cat([apt, mol], dim=1)
        
        # Shared encoding
        shared = self.shared_encoder(x)
        
        # Task-specific heads
        cls_logits = self.classifier(shared).squeeze(-1)
        reg_pred = self.regressor(shared).squeeze(-1)
        
        return cls_logits, reg_pred


print("✓ Scenario 4 model defined")


### 4.5. Model Visualization and Parameter Counting


In [ ]:
# Create sample models for visualization
# We'll use example dimensions based on OneHot + MorganFP
sample_apt_dim = 865  # OneHot: 216*4 + 1 = 865
sample_mol_dim = 2048  # MorganFP: 2048

# Create models with sample dimensions
models_to_visualize = [
    (PretrainedEncoderModel, "PretrainedEncoderModel", sample_apt_dim, sample_mol_dim, 
     {"hidden_dims": [512, 256, 128, 64], "dropout": 0.3}),
    (TwoTowerMLP, "TwoTowerMLP", sample_apt_dim, sample_mol_dim,
     {"tower_hidden_dims": [1024, 512, 256, 128], "fusion_hidden_dims": [512, 256, 128, 64], "dropout": 0.3}),
    (CrossAttentionModel, "CrossAttentionModel", sample_apt_dim, sample_mol_dim,
     {"hidden_dim": 512, "n_attention_layers": 4, "n_heads": 16, "dropout": 0.3}),
    (MultiTaskModel, "MultiTaskModel", sample_apt_dim, sample_mol_dim,
     {"shared_hidden_dims": [1024, 512, 256, 128], "task_hidden_dims": [256, 128, 64], "dropout": 0.3}),
]

print("Creating and visualizing model architectures...\n")
os.makedirs(os.path.join(ARTIFACTS_DIR, "figs"), exist_ok=True)

model_params_summary = []

for model_class, model_name, apt_dim, mol_dim, kwargs in models_to_visualize:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    
    # Create model
    model = model_class(apt_dim, mol_dim, **kwargs).to(device)
    
    # Count parameters
    params = count_parameters(model)
    model_params_summary.append({
        "Model": model_name,
        "Total Parameters": format_parameter_count(params['total']),
        "Trainable Parameters": format_parameter_count(params['trainable']),
        "Non-trainable Parameters": format_parameter_count(params['non_trainable']),
        "Total (raw)": params['total'],
        "Trainable (raw)": params['trainable']
    })
    
    print(f"Total parameters: {format_parameter_count(params['total'])}")
    print(f"Trainable parameters: {format_parameter_count(params['trainable'])}")
    
    # Visualize architecture
    save_path = os.path.join(ARTIFACTS_DIR, "figs", f"{model_name.lower()}_architecture.png")
    visualize_model_architecture(model, model_name, apt_dim, mol_dim, save_path=save_path)
    
    # Print model summary if torchinfo is available
    if TORCHINFO_AVAILABLE:
        print(f"\nDetailed model summary:")
        try:
            summary(model, input_size=[(1, apt_dim), (1, mol_dim)], device=device.type)
        except:
            print("  (Could not generate detailed summary)")

# Create summary table
params_df = pd.DataFrame(model_params_summary)
params_df = params_df.sort_values("Total (raw)", ascending=False)
print(f"\n{'='*60}")
print("Model Parameters Summary")
print(f"{'='*60}")
print(params_df[["Model", "Total Parameters", "Trainable Parameters"]].to_string(index=False))

# Save summary
params_summary_path = os.path.join(ARTIFACTS_DIR, "metrics", "model_parameters_summary.csv")
params_df.to_csv(params_summary_path, index=False)
print(f"\n✓ Parameters summary saved to: {params_summary_path}")


### 6.1. Advanced Encoder Configurations (for Scenario 1)


In [ ]:
# Generate advanced features for Scenario 1 (pretrained encoders)
# These will be used for more sophisticated models

print("Generating advanced features for pretrained encoders...")

# Advanced aptamer encoders
advanced_apt_cfgs = [
    {"name": "GENA", "func": gena_embed, "kwargs": {"batch_size": 32}},
    {"name": "DNABERT2", "func": dnabert2_embed, "kwargs": {"batch_size": 16}},
]

# Advanced molecule encoders
advanced_mol_cfgs = [
    {"name": "ChemBERTa", "func": chemberta_embed, "kwargs": {"batch_size": 32}},
    {"name": "UniMol", "func": unimol_embed, "kwargs": {}},
    {"name": "MolFormer", "func": molformer_embed, "kwargs": {"batch_size": 16}},
]

# Precompute advanced features (optional - can be slow)
# Uncomment to use pretrained embeddings
advanced_apt_features_map = {}
advanced_mol_features_map = {}

for cfg in advanced_apt_cfgs:
    name = cfg["name"]
    print(f"  Computing {name}...")
    try:
        features = cfg["func"](df["sequence"].tolist(), **cfg.get("kwargs", {}))
        advanced_apt_features_map[name] = features
        print(f"    Shape: {features.shape}")
    except Exception as e:
        print(f"    Error: {e}")

for cfg in advanced_mol_cfgs:
    name = cfg["name"]
    print(f"  Computing {name}...")
    try:
        features = cfg["func"](df["canonical_smiles"].tolist(), **cfg.get("kwargs", {}))
        advanced_mol_features_map[name] = features
        print(f"    Shape: {features.shape}")
    except Exception as e:
        print(f"    Error: {e}")

print("\n✓ Advanced encoder configurations ready (commented out to save time)")
print("  Uncomment the code above to use pretrained embeddings (GENA, DNABERT2, ChemBERTa, etc.)")


## 5. Training Utilities


## 8. Optuna Hyperparameter Optimization


In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device, multitask=False, alpha=1.0, beta=1.0):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    
    for batch in dataloader:
        apt = batch['apt'].to(device)
        mol = batch['mol'].to(device)
        label_class = batch['label_class'].to(device)
        
        optimizer.zero_grad()
        
        if multitask:
            cls_logits, reg_pred = model(apt, mol)
            label_reg = batch['label_reg'].to(device)
            
            # Combined loss
            cls_loss = criterion['cls'](cls_logits, label_class.float())
            reg_loss = criterion['reg'](reg_pred, label_reg)
            loss = alpha * cls_loss + beta * reg_loss
        else:
            logits = model(apt, mol)
            loss = criterion(logits, label_class.float())
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
    
    return total_loss / max(n_batches, 1)


def train_with_history(model, train_loader, val_loader, optimizer, criterion, device, 
                       n_epochs=50, patience=10, multitask=False, alpha=1.0, beta=1.0):
    """
    Train model with full history tracking for visualization.
    
    Returns:
        history: dict with 'train_loss', 'val_loss', 'val_mcc', 'val_roc_auc', etc.
    """
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_mcc': [],
        'val_roc_auc': [],
        'val_f1': []
    }
    
    best_val_score = -np.inf
    patience_counter = 0
    best_state_dict = None
    
    for epoch in range(n_epochs):
        # Training
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device, 
                                multitask=multitask, alpha=alpha, beta=beta)
        
        # Validation
        model.eval()
        val_loss = 0.0
        n_val_batches = 0
        all_preds_class = []
        all_scores = []
        all_labels_class = []
        
        with torch.no_grad():
            for batch in val_loader:
                apt = batch['apt'].to(device)
                mol = batch['mol'].to(device)
                label_class = batch['label_class'].to(device)
                
                if multitask:
                    cls_logits, reg_pred = model(apt, mol)
                    label_reg = batch['label_reg'].to(device)
                    cls_loss = criterion['cls'](cls_logits, label_class.float())
                    reg_loss = criterion['reg'](reg_pred, label_reg)
                    loss = alpha * cls_loss + beta * reg_loss
                    scores = torch.sigmoid(cls_logits).cpu().numpy()
                else:
                    logits = model(apt, mol)
                    loss = criterion(logits, label_class.float())
                    scores = torch.sigmoid(logits).cpu().numpy()
                
                val_loss += loss.item()
                n_val_batches += 1
                
                preds_class = (scores > 0.5).astype(int)
                all_preds_class.extend(preds_class)
                all_scores.extend(scores)
                all_labels_class.extend(label_class.cpu().numpy())
        
        val_loss = val_loss / max(n_val_batches, 1)
        val_metrics = compute_classification_metrics(
            np.array(all_labels_class),
            np.array(all_preds_class),
            np.array(all_scores)
        )
        
        # Update history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mcc'].append(val_metrics.get('MCC', np.nan))
        history['val_roc_auc'].append(val_metrics.get('ROC-AUC', np.nan))
        history['val_f1'].append(val_metrics.get('F1', np.nan))
        
        # Early stopping
        val_score = val_metrics.get('MCC', -np.inf)
        if val_score > best_val_score:
            best_val_score = val_score
            patience_counter = 0
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    
    return history


def evaluate(model, dataloader, device, multitask=False):
    """Evaluate model and return predictions and metrics."""
    model.eval()
    all_preds_class = []
    all_scores = []
    all_preds_reg = []
    all_labels_class = []
    all_labels_reg = []
    
    with torch.no_grad():
        for batch in dataloader:
            apt = batch['apt'].to(device)
            mol = batch['mol'].to(device)
            label_class = batch['label_class'].cpu().numpy()
            
            if multitask:
                cls_logits, reg_pred = model(apt, mol)
                scores = torch.sigmoid(cls_logits).cpu().numpy()
                preds_class = (scores > 0.5).astype(int)
                preds_reg = reg_pred.cpu().numpy()
                
                all_preds_reg.extend(preds_reg)
                if 'label_reg' in batch:
                    all_labels_reg.extend(batch['label_reg'].cpu().numpy())
            else:
                logits = model(apt, mol)
                scores = torch.sigmoid(logits).cpu().numpy()
                preds_class = (scores > 0.5).astype(int)
            
            all_preds_class.extend(preds_class)
            all_scores.extend(scores)
            all_labels_class.extend(label_class)
    
    results = {
        'y_true': np.array(all_labels_class),
        'y_pred': np.array(all_preds_class),
        'y_scores': np.array(all_scores)
    }
    
    if multitask and len(all_preds_reg) > 0:
        results['y_pred_reg'] = np.array(all_preds_reg)
        if len(all_labels_reg) > 0:
            results['y_true_reg'] = np.array(all_labels_reg)
    
    return results


print("✓ Training utilities defined")


## 6. Baseline Model Configurations


In [ ]:
# Load precomputed embeddings from ../artifacts/cache/embeddings
print("Loading precomputed embeddings from ../artifacts/cache/embeddings...")

# Define all encoder configurations (for loading)
apt_cfgs = [
    {"name": "OneHot"},
    {"name": "Kmer3"},
    {"name": "Kmer4"},
    {"name": "GENA"},
    {"name": "DNABERT2"},
]

mol_cfgs = [
    {"name": "MorganFP"},
    {"name": "MACCS"},
    {"name": "PhysChem"},
    {"name": "ChemBERTa"},
    {"name": "UniMol"},
    {"name": "MolFormer"},
    {"name": "CLAMP"},
]

# Try to load precomputed embeddings
apt_features_map = {}
mol_features_map = {}

loaded_embeddings = load_embeddings(df, apt_cfgs, mol_cfgs, embeddings_dir="../artifacts/cache/embeddings")

if loaded_embeddings is not None:
    apt_features_map, mol_features_map = loaded_embeddings
    print("\n✓ Successfully loaded precomputed embeddings!")
    print(f"  Aptamer encoders: {list(apt_features_map.keys())}")
    print(f"  Molecule encoders: {list(mol_features_map.keys())}")
    for name, feat in apt_features_map.items():
        print(f"    {name}: shape {feat.shape}")
    for name, feat in mol_features_map.items():
        print(f"    {name}: shape {feat.shape}")
else:
    print("\n⚠ Precomputed embeddings not found. Computing features on-the-fly...")
    # Fallback: compute features if embeddings not found
    try:
        apt_cfgs_compute = [
            {"name": "OneHot", "func": onehot_with_type_bit},
            {"name": "Kmer3", "func": kmer_freq_with_type_bit, "kwargs": {"k": 3}},
            {"name": "Kmer4", "func": kmer_freq_with_type_bit, "kwargs": {"k": 4}},
            {"name": "GENA", "func": gena_embed, "kwargs": {"batch_size": 32}},
            {"name": "DNABERT2", "func": dnabert2_embed, "kwargs": {"batch_size": 16}},
        ]
        
        mol_cfgs_compute = [
            {"name": "MorganFP", "func": morgan_fp},
            {"name": "MACCS", "func": maccs_fp},
            {"name": "PhysChem", "func": physchem_descriptors},
            {"name": "ChemBERTa", "func": chemberta_embed, "kwargs": {"batch_size": 32}},
            {"name": "UniMol", "func": unimol_embed, "kwargs": {}},
            {"name": "MolFormer", "func": molformer_embed, "kwargs": {"batch_size": 16}},
        ]
        
        print("\n=== Computing Aptamer Features ===")
        for cfg in apt_cfgs_compute:
            name = cfg["name"]
            print(f"  Computing {name}...", end=" ")
            try:
                features = cfg["func"](df["sequence"].tolist(), **cfg.get("kwargs", {}))
                apt_features_map[name] = features
                print(f"✓ Shape: {features.shape}")
            except Exception as e:
                print(f"✗ Error: {e}")
        
        print("\n=== Computing Molecule Features ===")
        for cfg in mol_cfgs_compute:
            name = cfg["name"]
            print(f"  Computing {name}...", end=" ")
            try:
                features = cfg["func"](df["canonical_smiles"].tolist(), **cfg.get("kwargs", {}))
                mol_features_map[name] = features
                print(f"✓ Shape: {features.shape}")
            except Exception as e:
                print(f"✗ Error: {e}")
    except Exception as e:
        print(f"✗ Error computing features: {e}")

print(f"\n✓ Final: {len(apt_features_map)} aptamer encoders, {len(mol_features_map)} molecule encoders")


## 7. Baseline Model Evaluation


In [ ]:
def run_baseline_evaluation(
    model_class,
    apt_features,
    mol_features,
    y_class,
    y_reg,
    splits,
    model_kwargs=None,
    n_epochs=50,
    batch_size=64,
    lr=1e-3,
    multitask=False,
    alpha=1.0,
    beta=1.0
):
    """
    Run baseline evaluation for a model configuration.
    
    Returns:
        List of metric dicts (one per fold)
    """
    if model_kwargs is None:
        model_kwargs = {}
    
    apt_dim = apt_features.shape[1]
    mol_dim = mol_features.shape[1]
    
    all_fold_metrics = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        print(f"  Fold {fold_idx + 1}/{len(splits)}...", end=" ")
        
        # Split data
        apt_train, apt_val = apt_features[train_idx], apt_features[val_idx]
        mol_train, mol_val = mol_features[train_idx], mol_features[val_idx]
        y_class_train, y_class_val = y_class[train_idx], y_class[val_idx]
        y_reg_train = y_reg[train_idx] if y_reg is not None else None
        y_reg_val = y_reg[val_idx] if y_reg is not None else None
        
                # For multitask models, filter out NaN values in y_reg
        if multitask and y_reg_train is not None:
            # Filter train set
            valid_train_mask = ~np.isnan(y_reg_train)
            apt_train = apt_train[valid_train_mask]
            mol_train = mol_train[valid_train_mask]
            y_class_train = y_class_train[valid_train_mask]
            y_reg_train = y_reg_train[valid_train_mask]
            
            # Keep full validation set for fair classification comparison
            # (regression metrics are filtered later).
        
        # Normalize features
        scaler_apt = StandardScaler()
        scaler_mol = StandardScaler()
        apt_train = scaler_apt.fit_transform(apt_train)
        apt_val = scaler_apt.transform(apt_val)
        mol_train = scaler_mol.fit_transform(mol_train)
        mol_val = scaler_mol.transform(mol_val)
        
        # Create datasets
        train_dataset = AptamerMoleculeDataset(
            apt_train, mol_train, y_class_train, y_reg_train
        )
        val_dataset = AptamerMoleculeDataset(
            apt_val, mol_val, y_class_val, y_reg_val
        )
        
        # Use drop_last=True to avoid BatchNorm error with batch_size=1
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Create model
        model = model_class(apt_dim, mol_dim, **model_kwargs).to(device)
        
        # Optimizer and loss
        optimizer = Adam(model.parameters(), lr=lr)
        if multitask:
            criterion = {
                'cls': nn.BCEWithLogitsLoss(),
                'reg': nn.MSELoss()
            }
        else:
            criterion = nn.BCEWithLogitsLoss()
        
        # Training loop
        best_val_score = -np.inf
        patience = 10
        patience_counter = 0
        best_state_dict = None
        
        for epoch in range(n_epochs):
            train_loss = train_epoch(
                model, train_loader, optimizer, criterion, device,
                multitask=multitask, alpha=alpha, beta=beta
            )
            
            # Validation
            val_results = evaluate(model, val_loader, device, multitask=multitask)
            val_metrics = compute_classification_metrics(
                val_results['y_true'],
                val_results['y_pred'],
                val_results['y_scores']
            )
            
            # Use MCC as validation score
            val_score = val_metrics['MCC']
            
            if val_score > best_val_score:
                best_val_score = val_score
                patience_counter = 0
                best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    break
        
        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)
        
        # Final evaluation
        val_results = evaluate(model, val_loader, device, multitask=multitask)
        fold_metrics = compute_classification_metrics(
            val_results['y_true'],
            val_results['y_pred'],
            val_results['y_scores']
        )
        
        # Add regression metrics if multitask
        if multitask and 'y_pred_reg' in val_results and 'y_true_reg' in val_results:
            reg_mask = ~np.isnan(val_results['y_true_reg'])
            if reg_mask.any():
                reg_metrics = compute_regression_metrics(
                    val_results['y_true_reg'][reg_mask],
                    val_results['y_pred_reg'][reg_mask]
                )
            else:
                reg_metrics = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
            fold_metrics.update(reg_metrics)
        
        all_fold_metrics.append(fold_metrics)
        print(f"MCC: {fold_metrics['MCC']:.3f}, ROC-AUC: {fold_metrics['ROC-AUC']:.3f}")
    
    return all_fold_metrics


print("✓ Baseline evaluation function defined")


In [ ]:
# Add TwoTower models with all combinations of top heads for aptamer and molecule
# Top head types: identity, cnn, lstm, transformer
top_head_types = ["identity", "cnn", "lstm", "transformer"]

# Base kwargs for all TwoTower models
base_two_tower_kwargs = {
    "seq_len": 32,
    "hidden_dim": 256,
    "output_dim": 128,
    "fusion_hidden_dims": [512, 256, 128, 64],
    "dropout": 0.3
}

# Top head specific kwargs
top_head_specific_kwargs = {
    "identity": {},
    "cnn": {"apt_top_kwargs": {"kernel_sizes": (3, 5, 7)}, 
            "mol_top_kwargs": {"kernel_sizes": (3, 5, 7)}},
    "lstm": {"apt_top_kwargs": {"num_layers": 2, "bidirectional": True},
             "mol_top_kwargs": {"num_layers": 2, "bidirectional": True}},
    "transformer": {"apt_top_kwargs": {"num_layers": 2, "num_heads": 8},
                    "mol_top_kwargs": {"num_layers": 2, "num_heads": 8}}
}

# Generate all combinations of aptamer and molecule top heads
top_head_configs = []
for apt_top in top_head_types:
    for mol_top in top_head_types:
        config_name = f"TwoTower_{apt_top.capitalize()}_{mol_top.capitalize()}"
        
        # Base kwargs
        kwargs = base_two_tower_kwargs.copy()
        kwargs["apt_top_type"] = apt_top
        kwargs["mol_top_type"] = mol_top
        
        # Add specific kwargs for each top head type
        if apt_top in top_head_specific_kwargs and "apt_top_kwargs" in top_head_specific_kwargs[apt_top]:
            kwargs["apt_top_kwargs"] = top_head_specific_kwargs[apt_top]["apt_top_kwargs"]
        if mol_top in top_head_specific_kwargs and "mol_top_kwargs" in top_head_specific_kwargs[mol_top]:
            kwargs["mol_top_kwargs"] = top_head_specific_kwargs[mol_top]["mol_top_kwargs"]
        
        top_head_configs.append({
            "name": config_name,
            "class": TwoTowerWithTopHeads,
            "kwargs": kwargs,
            "multitask": False
        })

# Combine all model configurations
model_configs = []
model_configs = model_configs + top_head_configs
print(f"✓ Added {len(top_head_configs)} TwoTower models with all top head combinations")
print(f"  Combinations: {len(top_head_types)} aptamer × {len(top_head_types)} molecule = {len(top_head_configs)}")
print(f"  Total models: {len(model_configs)}")


In [ ]:
# ============================================================
# UTILITY FUNCTIONS FOR INCREMENTAL SAVING AND RESUME
# ============================================================

def load_existing_results(results_path, histories_path):
    """
    Load existing screening results and histories if they exist.
    
    Returns:
        (screening_results, all_histories, completed_configs)
    """
    screening_results = []
    all_histories = {}
    completed_configs = set()
    
    # Load results CSV
    if os.path.exists(results_path):
        try:
            existing_df = pd.read_csv(results_path)
            screening_results = existing_df.to_dict('records')
            completed_configs = {row['config'] for row in screening_results if 'config' in row}
            print(f"✓ Loaded {len(screening_results)} existing results from {results_path}")
        except Exception as e:
            print(f"⚠ Could not load existing results: {e}")
    
    # Load histories JSON
    if os.path.exists(histories_path):
        try:
            with open(histories_path, 'r') as f:
                all_histories = json.load(f)
                # Convert lists back to proper format
                for key, value in all_histories.items():
                    if isinstance(value, list):
                        all_histories[key] = value
            print(f"✓ Loaded {len(all_histories)} existing histories from {histories_path}")
        except Exception as e:
            print(f"⚠ Could not load existing histories: {e}")
    
    return screening_results, all_histories, completed_configs


def save_results_incremental(screening_results, all_histories, results_path, histories_path):
    """
    Save screening results and histories incrementally.
    
    Args:
        screening_results: List of result dictionaries
        all_histories: Dictionary of histories (will be updated with current split's histories)
        results_path: Path to save CSV results
        histories_path: Path to save JSON histories
    """
    try:
        # Save results CSV
        if len(screening_results) > 0:
            df = pd.DataFrame(screening_results)
            df.to_csv(results_path, index=False)
        
        # Save histories JSON (convert numpy arrays to lists for JSON serialization)
        histories_serializable = {}
        for key, value in all_histories.items():
            if isinstance(value, list):
                # Convert each fold history
                serialized_folds = []
                for fold_history in value:
                    serialized_fold = {}
                    for metric_name, metric_values in fold_history.items():
                        if isinstance(metric_values, list):
                            # Convert numpy arrays to lists
                            serialized_fold[metric_name] = [
                                float(v) if not np.isnan(v) else None 
                                for v in metric_values
                            ]
                        else:
                            serialized_fold[metric_name] = metric_values
                    serialized_folds.append(serialized_fold)
                histories_serializable[key] = serialized_folds
        
        with open(histories_path, 'w') as f:
            json.dump(histories_serializable, f, indent=2)
            
    except Exception as e:
        print(f"⚠ Error saving results: {e}")


def clear_memory():
    """Clear GPU and CPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


print("✓ Utility functions for incremental saving and memory management defined")


In [ ]:
# Updated run_baseline_evaluation with history tracking support
# This overrides the previous definition to add track_history parameter

def run_baseline_evaluation_with_history(
    model_class,
    apt_features,
    mol_features,
    y_class,
    y_reg,
    splits,
    model_kwargs=None,
    n_epochs=50,
    batch_size=64,
    lr=1e-3,
    multitask=False,
    alpha=1.0,
    beta=1.0,
    track_history=True,
    patience=15
):
    """
    Run baseline evaluation for a model configuration with training history tracking.
    
    Returns:
        all_fold_metrics: List of metric dicts (one per fold)
        all_fold_histories: List of training histories (one per fold) if track_history=True
    """
    if model_kwargs is None:
        model_kwargs = {}
    
    apt_dim = apt_features.shape[1]
    mol_dim = mol_features.shape[1]
    
    all_fold_metrics = []
    all_fold_histories = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        print(f"  Fold {fold_idx + 1}/{len(splits)}...", end=" ")
        
        # Split data
        apt_train, apt_val = apt_features[train_idx], apt_features[val_idx]
        mol_train, mol_val = mol_features[train_idx], mol_features[val_idx]
        y_class_train, y_class_val = y_class[train_idx], y_class[val_idx]
        y_reg_train = y_reg[train_idx] if y_reg is not None else None
        y_reg_val = y_reg[val_idx] if y_reg is not None else None
        
        # For multitask models, filter out NaN values in y_reg
        if multitask and y_reg_train is not None:
            # Filter train set
            valid_train_mask = ~np.isnan(y_reg_train)
            apt_train = apt_train[valid_train_mask]
            mol_train = mol_train[valid_train_mask]
            y_class_train = y_class_train[valid_train_mask]
            y_reg_train = y_reg_train[valid_train_mask]
            
            # Keep full validation set for fair classification comparison
            # (regression metrics are filtered later).
        
        # Normalize features
        scaler_apt = StandardScaler()
        scaler_mol = StandardScaler()
        apt_train = scaler_apt.fit_transform(apt_train)
        apt_val = scaler_apt.transform(apt_val)
        mol_train = scaler_mol.fit_transform(mol_train)
        mol_val = scaler_mol.transform(mol_val)
        
        # Create datasets
        train_dataset = AptamerMoleculeDataset(
            apt_train, mol_train, y_class_train, y_reg_train
        )
        val_dataset = AptamerMoleculeDataset(
            apt_val, mol_val, y_class_val, y_reg_val
        )
        
        # Use drop_last=True to avoid BatchNorm error with batch_size=1
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Create model
        model = model_class(apt_dim, mol_dim, **model_kwargs).to(device)
        
        # Optimizer and loss
        optimizer = Adam(model.parameters(), lr=lr)
        if multitask:
            criterion = {
                'cls': nn.BCEWithLogitsLoss(),
                'reg': nn.MSELoss()
            }
        else:
            criterion = nn.BCEWithLogitsLoss()
        
        # Training with history tracking
        if track_history:
            history = train_with_history(
                model, train_loader, val_loader, optimizer, criterion, device,
                n_epochs=n_epochs, patience=patience, multitask=multitask, alpha=alpha, beta=beta
            )
            all_fold_histories.append(history)
        else:
            # Training loop without history
            best_val_score = -np.inf
            patience_counter = 0
            best_state_dict = None
            
            for epoch in range(n_epochs):
                train_loss = train_epoch(
                    model, train_loader, optimizer, criterion, device,
                    multitask=multitask, alpha=alpha, beta=beta
                )
                
                # Validation
                val_results = evaluate(model, val_loader, device, multitask=multitask)
                val_metrics = compute_classification_metrics(
                    val_results['y_true'],
                    val_results['y_pred'],
                    val_results['y_scores']
                )
                
                # Use MCC as validation score
                val_score = val_metrics['MCC']
                
                if val_score > best_val_score:
                    best_val_score = val_score
                    patience_counter = 0
                    best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        break
            if best_state_dict is not None:
                model.load_state_dict(best_state_dict)
        
        # Final evaluation
        val_results = evaluate(model, val_loader, device, multitask=multitask)
        fold_metrics = compute_classification_metrics(
            val_results['y_true'],
            val_results['y_pred'],
            val_results['y_scores']
        )
        
        # Add regression metrics if multitask
        if multitask and 'y_pred_reg' in val_results and 'y_true_reg' in val_results:
            reg_mask = ~np.isnan(val_results['y_true_reg'])
            if reg_mask.any():
                reg_metrics = compute_regression_metrics(
                    val_results['y_true_reg'][reg_mask],
                    val_results['y_pred_reg'][reg_mask]
                )
            else:
                reg_metrics = {'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan}
            fold_metrics.update(reg_metrics)
        
        all_fold_metrics.append(fold_metrics)
        print(f"MCC: {fold_metrics['MCC']:.3f}")
    
    if track_history:
        return all_fold_metrics, all_fold_histories
    else:
        return all_fold_metrics

# Override the original function
run_baseline_evaluation = run_baseline_evaluation_with_history

print("✓ Updated run_baseline_evaluation with history tracking support")


### 7.1. Run Baseline Models


In [ ]:
# ============================================================
# COMPREHENSIVE SCREENING WITH INCREMENTAL SAVING AND MULTIPLE SPLITS
# ============================================================

# Define split modes to evaluate
split_modes = ["stratified", "disjoint_aptamer", "disjoint_molecule"]

# Select top encoders to reduce combinations (20-30 per split)
top_apt_encoders = ["OneHot", "GENA"]  # Top 4
top_mol_encoders = ["ChemBERTa", "UniMol", "CLAMP"]  # Top 4

# Filter feature maps
filtered_apt_features_map = {k: v for k, v in apt_features_map.items() if k in top_apt_encoders}
filtered_mol_features_map = {k: v for k, v in mol_features_map.items() if k in top_mol_encoders}

# Select top models (7 best performing)
top_model_configs = [
    {
        "name": "TwoTowerMLP",
        "class": TwoTowerMLP,
        "kwargs": {
            "tower_hidden_dims": [1024, 512, 256, 128],
            "fusion_hidden_dims": [512, 256, 128, 64],
            "dropout": 0.3
        },
        "multitask": False
    },
    {
        "name": "CrossAttention",
        "class": CrossAttentionModel,
        "kwargs": {
            "hidden_dim": 512,
            "n_attention_layers": 4,
            "n_heads": 16,
            "dropout": 0.3,
            "classifier_hidden_dims": [512, 256, 128]
        },
        "multitask": False
    },
    {
        "name": "MultiTask",
        "class": MultiTaskModel,
        "kwargs": {
            "shared_hidden_dims": [1024, 512, 256, 128],
            "task_hidden_dims": [256, 128, 64],
            "dropout": 0.3
        },
        "multitask": True,
        "alpha": 1.0,
        "beta": 0.1
    }
]

# Add best TwoTower top head combinations
top_two_tower_configs = [
    {
        "name": "TwoTower_CNN_CNN",
        "class": TwoTowerWithTopHeads,
        "kwargs": {
            "apt_top_type": "cnn",
            "mol_top_type": "cnn",
            "seq_len": 32,
            "hidden_dim": 256,
            "output_dim": 128,
            "fusion_hidden_dims": [512, 256, 128, 64],
            "dropout": 0.3,
            "apt_top_kwargs": {"kernel_sizes": (3, 5, 7)},
            "mol_top_kwargs": {"kernel_sizes": (3, 5, 7)}
        },
        "multitask": False
    },
    {
        "name": "TwoTower_LSTM_LSTM",
        "class": TwoTowerWithTopHeads,
        "kwargs": {
            "apt_top_type": "lstm",
            "mol_top_type": "lstm",
            "seq_len": 32,
            "hidden_dim": 256,
            "output_dim": 128,
            "fusion_hidden_dims": [512, 256, 128, 64],
            "dropout": 0.3,
            "apt_top_kwargs": {"num_layers": 2, "bidirectional": True},
            "mol_top_kwargs": {"num_layers": 2, "bidirectional": True}
        },
        "multitask": False
    },
    {
        "name": "TwoTower_Transformer_Transformer",
        "class": TwoTowerWithTopHeads,
        "kwargs": {
            "apt_top_type": "transformer",
            "mol_top_type": "transformer",
            "seq_len": 32,
            "hidden_dim": 256,
            "output_dim": 128,
            "fusion_hidden_dims": [512, 256, 128, 64],
            "dropout": 0.3,
            "apt_top_kwargs": {"num_layers": 2, "num_heads": 8},
            "mol_top_kwargs": {"num_layers": 2, "num_heads": 8}
        },
        "multitask": False
    }
]

model_configs = top_model_configs + top_two_tower_configs

print(f"Selected {len(model_configs)} models")
print(f"Selected {len(filtered_apt_features_map)} aptamer encoders: {list(filtered_apt_features_map.keys())}")
print(f"Selected {len(filtered_mol_features_map)} molecule encoders: {list(filtered_mol_features_map.keys())}")
print(f"Total combinations per split: {len(model_configs) * len(filtered_apt_features_map) * len(filtered_mol_features_map)}")
print(f"Total splits: {len(split_modes)}")

# Training parameters
n_epochs = 50
patience = 15
batch_size = 64
lr = 1e-3

# Store all results across all splits
all_screening_results = []
all_histories = {}

# Iterate through all split modes
for split_mode in split_modes:
    print(f"\n{'='*80}")
    print(f"SPLIT MODE: {split_mode.upper()}")
    print(f"{'='*80}")
    
    # Load splits for this mode
    try:
        splits = load_splits(split_mode, base_dir=SPLITS_DIR)
        print(f"Loaded {len(splits)} folds for {split_mode} split")
    except Exception as e:
        print(f"⚠ Could not load splits for {split_mode}: {e}")
        continue
    
    # Define paths for incremental saving (per split mode)
    screening_results_path = os.path.join(ARTIFACTS_DIR, "metrics", f"comprehensive_screening_{split_mode}.csv")
    screening_histories_path = os.path.join(ARTIFACTS_DIR, "metrics", f"comprehensive_screening_{split_mode}_histories.json")
    
    # Load existing results if available (for resume capability)
    print(f"\nLoading existing results for {split_mode}...")
    screening_results, split_histories, completed_configs = load_existing_results(
        screening_results_path, screening_histories_path
    )
    
    # Add split_mode to completed configs to make them unique
    completed_configs = {f"{split_mode}::{config}" for config in completed_configs}
    
    if len(completed_configs) > 0:
        print(f"  Found {len(completed_configs)} already completed configurations for {split_mode}")
    
    # Update all_histories with split-specific histories
    for key, value in split_histories.items():
        all_histories[f"{split_mode}::{key}"] = value
    
    total_combinations = len(model_configs) * len(filtered_apt_features_map) * len(filtered_mol_features_map)
    current_combination = 0
    
    print(f"\n{'='*80}")
    print(f"COMPREHENSIVE SCREENING: {split_mode.upper()}")
    print(f"{'='*80}")
    print(f"Models: {len(model_configs)}")
    print(f"Aptamer encoders: {len(filtered_apt_features_map)}")
    print(f"Molecule encoders: {len(filtered_mol_features_map)}")
    print(f"Total combinations: {total_combinations}")
    print(f"Already completed: {len(completed_configs)}")
    print(f"Remaining: {total_combinations - len(completed_configs)}")
    print(f"{'='*80}\n")
    
    # Iterate through all model architectures
    for model_cfg in model_configs:
        model_name = model_cfg["name"]
        model_class = model_cfg["class"]
        model_kwargs = model_cfg["kwargs"]
        is_multitask = model_cfg.get("multitask", False)
        alpha = model_cfg.get("alpha", 1.0)
        beta = model_cfg.get("beta", 1.0)
        
        print(f"\n{'#'*80}")
        print(f"# MODEL: {model_name}")
        print(f"{'#'*80}\n")
        
        # Iterate through all encoder combinations
        for apt_name, apt_feat in filtered_apt_features_map.items():
            for mol_name, mol_feat in filtered_mol_features_map.items():
                current_combination += 1
                config_name = f"{model_name} | {apt_name} + {mol_name}"
                unique_config_name = f"{split_mode}::{config_name}"
                
                # Skip if already completed
                if unique_config_name in completed_configs:
                    print(f"\n[{current_combination}/{total_combinations}] {config_name} - SKIPPED (already completed)")
                    continue
                
                print(f"\n[{current_combination}/{total_combinations}] {config_name}")
                print(f"{'='*80}")
                
                model = None  # Initialize for cleanup
                try:
                    # Verify feature shapes
                    print(f"  Aptamer features: {apt_feat.shape}, Molecule features: {mol_feat.shape}")
                    
                    # Run evaluation with history tracking
                    metrics, histories = run_baseline_evaluation(
                        model_class,
                        apt_feat, mol_feat, y_class, y_reg,
                        splits,
                        model_kwargs=model_kwargs,
                        n_epochs=n_epochs,
                        batch_size=batch_size,
                        lr=lr,
                        multitask=is_multitask,
                        alpha=alpha,
                        beta=beta,
                        track_history=True,
                        patience=patience
                    )
                    
                    # Check if training actually ran (verify history length)
                    if histories and len(histories) > 0:
                        min_epochs = min(len(h['train_loss']) for h in histories)
                        max_epochs = max(len(h['train_loss']) for h in histories)
                        print(f"  Training epochs: {min_epochs}-{max_epochs} per fold")
                        
                        if min_epochs < 5:
                            print(f"  ⚠ WARNING: Training stopped very early (only {min_epochs} epochs)!")
                    
                    # Aggregate metrics
                    agg_metrics = aggregate_metrics(metrics)
                    
                    # Store results
                    result = {
                        "split_mode": split_mode,
                        "model": model_name,
                        "apt_encoder": apt_name,
                        "mol_encoder": mol_name,
                        "config": config_name,
                        **agg_metrics
                    }
                    screening_results.append(result)
                    all_screening_results.append(result)
                    
                    # Store histories for visualization
                    all_histories[unique_config_name] = histories
                    split_histories[config_name] = histories  # Also store in split-specific dict
                    
                    print(f"  ✓ MCC: {agg_metrics.get('MCC mean', np.nan):.3f} ± {agg_metrics.get('MCC std', np.nan):.3f}")
                    print(f"  ✓ ROC-AUC: {agg_metrics.get('ROC-AUC mean', np.nan):.3f} ± {agg_metrics.get('ROC-AUC std', np.nan):.3f}")
                    
                    # Save incrementally after each configuration
                    save_results_incremental(
                        screening_results, split_histories, 
                        screening_results_path, screening_histories_path
                    )
                    print(f"  💾 Saved results incrementally")
                    
                    # Clear memory to prevent overflow
                    try:
                        del model
                    except:
                        pass
                    clear_memory()
                    print(f"  🧹 Memory cleared")
                    
                except Exception as e:
                    print(f"  ✗ Error: {e}")
                    import traceback
                    traceback.print_exc()
                    
                    # Save what we have so far even on error
                    try:
                        save_results_incremental(
                            screening_results, split_histories, 
                            screening_results_path, screening_histories_path
                        )
                        print(f"  💾 Saved partial results after error")
                    except:
                        pass
                    
                    # Clear memory after error
                    try:
                        del model
                    except:
                        pass
                    clear_memory()
                    continue
    
    # Final save for this split mode
    try:
        save_results_incremental(
            screening_results, split_histories, 
            screening_results_path, screening_histories_path
        )
        print(f"\n✓ Final results saved for {split_mode}")
    except Exception as e:
        print(f"⚠ Error saving final results for {split_mode}: {e}")

# Create combined results DataFrame
if len(all_screening_results) > 0:
    all_screening_df = pd.DataFrame(all_screening_results)
    all_screening_df = all_screening_df.sort_values("MCC mean", ascending=False)
    
    print("\n" + "="*80)
    print("COMBINED SCREENING RESULTS SUMMARY (ALL SPLITS)")
    print("="*80)
    print(all_screening_df[["split_mode", "model", "apt_encoder", "mol_encoder", "MCC mean", "MCC std", 
                            "ROC-AUC mean", "ROC-AUC std", "F1 mean"]].to_string(index=False))
    
    # Save combined results
    combined_path = os.path.join(ARTIFACTS_DIR, "metrics", "comprehensive_screening_all_splits.csv")
    all_screening_df.to_csv(combined_path, index=False)
    print(f"\n✓ Combined results saved to: {combined_path}")
else:
    print("\n⚠ No results to save!")


In [ ]:
screening_df# = all_screening_df.copy()

In [ ]:
# ============================================================
# VISUALIZE TRAINING CURVES FOR ALL ENCODER COMBINATIONS
# ============================================================

def plot_training_curves(histories_dict, save_dir=None):
    """
    Plot training curves for all encoder configurations.
    
    Args:
        histories_dict: dict mapping config_name -> list of fold histories
        save_dir: directory to save plots (optional)
    """
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    
    n_configs = len(histories_dict)
    if n_configs == 0:
        print("No histories to plot")
        return
    
    # Create subplots: one row per configuration
    fig, axes = plt.subplots(n_configs, 2, figsize=(16, 4 * n_configs))
    if n_configs == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (config_name, fold_histories) in enumerate(histories_dict.items()):
        ax_loss = axes[idx, 0]
        ax_metrics = axes[idx, 1]
        
        # Aggregate histories across folds (mean ± std)
        max_epochs = max(len(h['train_loss']) for h in fold_histories)
        epochs = np.arange(1, max_epochs + 1)
        
        # Loss curves
        train_losses = []
        val_losses = []
        for h in fold_histories:
            train_losses.append(h['train_loss'] + [np.nan] * (max_epochs - len(h['train_loss'])))
            val_losses.append(h['val_loss'] + [np.nan] * (max_epochs - len(h['val_loss'])))
        
        train_losses = np.array(train_losses)
        val_losses = np.array(val_losses)
        
        train_loss_mean = np.nanmean(train_losses, axis=0)
        train_loss_std = np.nanstd(train_losses, axis=0)
        val_loss_mean = np.nanmean(val_losses, axis=0)
        val_loss_std = np.nanstd(val_losses, axis=0)
        
        # Plot loss
        ax_loss.plot(epochs, train_loss_mean, 'b-', label='Train Loss', linewidth=2)
        ax_loss.fill_between(epochs, train_loss_mean - train_loss_std, 
                             train_loss_mean + train_loss_std, alpha=0.2, color='blue')
        ax_loss.plot(epochs, val_loss_mean, 'r-', label='Val Loss', linewidth=2)
        ax_loss.fill_between(epochs, val_loss_mean - val_loss_std, 
                             val_loss_mean + val_loss_std, alpha=0.2, color='red')
        ax_loss.set_xlabel('Epoch', fontsize=10)
        ax_loss.set_ylabel('Loss', fontsize=10)
        ax_loss.set_title(f'{config_name} - Loss Curves', fontsize=12, fontweight='bold')
        ax_loss.legend()
        ax_loss.grid(True, alpha=0.3)
        
        # Metrics curves
        val_mccs = []
        val_roc_aucs = []
        for h in fold_histories:
            val_mccs.append(h['val_mcc'] + [np.nan] * (max_epochs - len(h['val_mcc'])))
            val_roc_aucs.append(h['val_roc_auc'] + [np.nan] * (max_epochs - len(h['val_roc_auc'])))
        
        val_mccs = np.array(val_mccs)
        val_roc_aucs = np.array(val_roc_aucs)
        
        val_mcc_mean = np.nanmean(val_mccs, axis=0)
        val_mcc_std = np.nanstd(val_mccs, axis=0)
        val_roc_auc_mean = np.nanmean(val_roc_aucs, axis=0)
        val_roc_auc_std = np.nanstd(val_roc_aucs, axis=0)
        
        # Plot metrics
        ax_metrics.plot(epochs, val_mcc_mean, 'g-', label='Val MCC', linewidth=2)
        ax_metrics.fill_between(epochs, val_mcc_mean - val_mcc_std, 
                                val_mcc_mean + val_mcc_std, alpha=0.2, color='green')
        ax_metrics.plot(epochs, val_roc_auc_mean, 'orange', label='Val ROC-AUC', linewidth=2)
        ax_metrics.fill_between(epochs, val_roc_auc_mean - val_roc_auc_std, 
                                val_roc_auc_mean + val_roc_auc_std, alpha=0.2, color='orange')
        ax_metrics.set_xlabel('Epoch', fontsize=10)
        ax_metrics.set_ylabel('Metric Value', fontsize=10)
        ax_metrics.set_title(f'{config_name} - Validation Metrics', fontsize=12, fontweight='bold')
        ax_metrics.legend()
        ax_metrics.grid(True, alpha=0.3)
        ax_metrics.set_ylim([0, 1])
    
    plt.tight_layout()
    
    if save_dir:
        save_path = os.path.join(save_dir, "training_curves_all_encoders.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Training curves saved to: {save_path}")
    
    plt.show()


# Plot all training curves
if len(all_histories) > 0:
    print("\n" + "="*80)
    print("VISUALIZING TRAINING CURVES")
    print("="*80)
    
    # If too many configurations, create separate plots per model
    if len(all_histories) > 20:
        print(f"  Creating separate plots per model (total: {len(all_histories)} configurations)...")
        
        # Group by model
        models = ["PretrainedEncoder", "TwoTowerMLP", "CrossAttention", "MultiTask"]
        for model_name in models:
            model_histories = {k: v for k, v in all_histories.items() if k.startswith(model_name)}
            if len(model_histories) > 0:
                save_path = os.path.join(ARTIFACTS_DIR, "figs", f"training_curves_{model_name.lower()}.png")
                plot_training_curves(model_histories, save_dir=None)
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"  ✓ Saved: {save_path}")
        
        # Also create overall summary plot (top 20 by MCC if available)
        if len(screening_df) > 0:
            top_configs = screening_df.head(20)["config"].tolist()
            top_histories = {k: v for k, v in all_histories.items() if k in top_configs}
            if len(top_histories) > 0:
                save_path = os.path.join(ARTIFACTS_DIR, "figs", "training_curves_top20.png")
                plot_training_curves(top_histories, save_dir=None)
                plt.savefig(save_path, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"  ✓ Saved: {save_path}")
    else:
        plot_training_curves(all_histories, save_dir=os.path.join(ARTIFACTS_DIR, "figs"))
else:
    print("No training histories available for visualization")


In [ ]:
# ============================================================
# ABLATION STUDY: Feature Importance Analysis
# ============================================================
# This study investigates the importance of different embedding components
# by replacing them with noise or shuffled labels

print("="*80)
print("ABLATION STUDY: TwoTowerMLP with GENA + UniMol")
print("="*80)

# Ensure we have the required embeddings
if "GENA" not in apt_features_map:
    print("⚠ GENA embeddings not found. Please run the embedding loading cell first.")
elif "UniMol" not in mol_features_map:
    print("⚠ UniMol embeddings not found. Please run the embedding loading cell first.")
else:
    # Get original embeddings
    apt_original = apt_features_map["GENA"].copy()
    mol_original = mol_features_map["UniMol"].copy()
    
    print(f"\nOriginal embedding shapes:")
    print(f"  Aptamer (GENA): {apt_original.shape}")
    print(f"  Molecule (UniMol): {mol_original.shape}")
    
    # Set random seed for reproducibility
    np.random.seed(RANDOM_STATE)
    
    # Load splits for evaluation
    splits = load_splits("stratified", base_dir="../dataset/splits")
    print(f"\nLoaded {len(splits)} folds for evaluation")
    
    # Define ablation scenarios
    ablation_scenarios = {
        "baseline": {
            "description": "Original embeddings (baseline)",
            "modify_apt": False,
            "modify_mol": False,
            "shuffle_last_apt": False
        },
        "noise_molecule": {
            "description": "Molecule embeddings replaced with random noise",
            "modify_apt": False,
            "modify_mol": True,
            "shuffle_last_apt": False
        },
        "noise_aptamer": {
            "description": "Aptamer embeddings (except last element) replaced with random noise",
            "modify_apt": True,
            "modify_mol": False,
            "shuffle_last_apt": False
        },
        "shuffle_last_apt": {
            "description": "Last element of aptamer embedding replaced with shuffled labels",
            "modify_apt": False,
            "modify_mol": False,
            "shuffle_last_apt": True
        },
        "noise_molecule_and_aptamer": {
            "description": "Both molecule and aptamer embeddings (except last) replaced with noise",
            "modify_apt": True,
            "modify_mol": True,
            "shuffle_last_apt": False
        },
        "all_ablation": {
            "description": "All ablations: noise for molecule, noise for aptamer (except last), shuffled last aptamer",
            "modify_apt": True,
            "modify_mol": True,
            "shuffle_last_apt": True
        }
    }
    
    # Store results
    ablation_results = []
    
    # Run each ablation scenario
    for scenario_name, scenario_config in ablation_scenarios.items():
        print(f"\n{'='*80}")
        print(f"SCENARIO: {scenario_name.upper()}")
        print(f"Description: {scenario_config['description']}")
        print(f"{'='*80}")
        
        # Create modified embeddings
        apt_modified = apt_original.copy()
        mol_modified = mol_original.copy()
        
        # Modify molecule embeddings: replace with random noise
        if scenario_config["modify_mol"]:
            # Generate random noise with same shape and similar statistics
            mol_mean = mol_original.mean(axis=0)
            mol_std = mol_original.std(axis=0)
            mol_modified = np.random.normal(mol_mean, mol_std, size=mol_original.shape)
            print(f"  ✓ Replaced molecule embeddings with random noise (mean={mol_mean.mean():.3f}, std={mol_std.mean():.3f})")
        
        # Modify aptamer embeddings: replace all except last element with random noise
        if scenario_config["modify_apt"]:
            # Generate random noise for all but last dimension
            apt_mean = apt_original[:, :-1].mean(axis=0)
            apt_std = apt_original[:, :-1].std(axis=0)
            apt_modified[:, :-1] = np.random.normal(apt_mean, apt_std, size=apt_original[:, :-1].shape)
            print(f"  ✓ Replaced aptamer embeddings (except last) with random noise")
        
        # Shuffle last element of aptamer embedding: replace with shuffled labels
        if scenario_config["shuffle_last_apt"]:
            # Get original labels
            shuffled_labels = y_class.copy()
            np.random.shuffle(shuffled_labels)
            # Replace last element of aptamer embedding with shuffled labels
            apt_modified[:, -1] = shuffled_labels.astype(float)
            print(f"  ✓ Replaced last element of aptamer embedding with shuffled labels")
        
        # Run evaluation
        print(f"\n  Running evaluation...")
        try:
            result = run_baseline_evaluation(
                TwoTowerMLP,
                apt_modified, mol_modified, y_class, y_reg,
                splits,
                model_kwargs={
                    "tower_hidden_dims": [1024, 512, 256, 128],
                    "fusion_hidden_dims": [512, 256, 128, 64],
                    "dropout": 0.3
                },
                n_epochs=50,
                batch_size=64,
                lr=1e-3,
                multitask=False,
                track_history=True,
                patience=15
            )
            
            # Handle return value (can be tuple or single value)
            if isinstance(result, tuple):
                metrics, histories = result
            else:
                metrics = result
                histories = None
            
            # Aggregate metrics
            agg_metrics = aggregate_metrics(metrics)
            
            # Store results
            result = {
                "scenario": scenario_name,
                "description": scenario_config["description"],
                **agg_metrics
            }
            ablation_results.append(result)
            
            print(f"\n  Results:")
            print(f"    MCC: {agg_metrics.get('MCC mean', np.nan):.3f} ± {agg_metrics.get('MCC std', np.nan):.3f}")
            print(f"    ROC-AUC: {agg_metrics.get('ROC-AUC mean', np.nan):.3f} ± {agg_metrics.get('ROC-AUC std', np.nan):.3f}")
            print(f"    F1: {agg_metrics.get('F1 mean', np.nan):.3f} ± {agg_metrics.get('F1 std', np.nan):.3f}")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
            import traceback
            traceback.print_exc()
            continue
        
        # Clear memory
        clear_memory()
    
    # Create results DataFrame
    if len(ablation_results) > 0:
        ablation_df = pd.DataFrame(ablation_results)
        ablation_df = ablation_df.sort_values("MCC mean", ascending=False)
        
        print(f"\n{'='*80}")
        print("ABLATION STUDY RESULTS SUMMARY")
        print(f"{'='*80}")
        print(ablation_df[["scenario", "description", "MCC mean", "MCC std", 
                          "ROC-AUC mean", "ROC-AUC std", "F1 mean", "F1 std"]].to_string(index=False))
        
        # Save results
        ablation_path = os.path.join(ARTIFACTS_DIR, "metrics", "ablation_study_gena_unimol.csv")
        ablation_df.to_csv(ablation_path, index=False)
        print(f"\n✓ Results saved to: {ablation_path}")
        
        # Calculate performance degradation
        if "baseline" in ablation_df["scenario"].values:
            baseline_mcc = ablation_df[ablation_df["scenario"] == "baseline"]["MCC mean"].values[0]
            print(f"\n{'='*80}")
            print("PERFORMANCE DEGRADATION (vs Baseline)")
            print(f"{'='*80}")
            for _, row in ablation_df.iterrows():
                if row["scenario"] != "baseline":
                    degradation = baseline_mcc - row["MCC mean"]
                    pct_degradation = (degradation / baseline_mcc) * 100
                    print(f"  {row['scenario']:30s}: {degradation:+.3f} ({pct_degradation:+.1f}%)")
    else:
        print("\n⚠ No results to display")

print("\n" + "="*80)
print("ABLATION STUDY COMPLETE")
print("="*80)


## Regression Model with Best Architecture


In [ ]:
# ============================================================
# REGRESSION MODEL WITH BEST ARCHITECTURE
# ============================================================
# Train a regression model using the best classification architecture
# (TwoTower_CNN_CNN with GENA + UniMol based on screening results)

print("="*80)
print("REGRESSION MODEL: TwoTower_CNN_CNN with GENA + UniMol")
print("="*80)

# Check if we have the required embeddings
if "GENA" not in apt_features_map:
    print("⚠ GENA embeddings not found. Please run the embedding loading cell first.")
elif "UniMol" not in mol_features_map:
    print("⚠ UniMol embeddings not found. Please run the embedding loading cell first.")
else:
    # Get embeddings
    apt_feat = apt_features_map["GENA"]
    mol_feat = mol_features_map["UniMol"]
    
    print(f"\nEmbedding shapes:")
    print(f"  Aptamer (GENA): {apt_feat.shape}")
    print(f"  Molecule (UniMol): {mol_feat.shape}")
    
    # Filter data to only include samples with pKd values
    valid_mask = ~np.isnan(y_reg)
    apt_feat_reg = apt_feat[valid_mask]
    mol_feat_reg = mol_feat[valid_mask]
    y_reg_filtered = y_reg[valid_mask]
    
    print(f"\nDataset size:")
    print(f"  Total samples: {len(y_reg)}")
    print(f"  Samples with pKd: {len(y_reg_filtered)} ({100 * len(y_reg_filtered) / len(y_reg):.1f}%)")
    print(f"  pKd range: [{y_reg_filtered.min():.2f}, {y_reg_filtered.max():.2f}]")
    print(f"  pKd mean ± std: {y_reg_filtered.mean():.2f} ± {y_reg_filtered.std():.2f}")
    
    # Store results for all split modes
    regression_results = []
    
    # Evaluate on all split modes
    split_modes = ["stratified", "disjoint_aptamer", "disjoint_molecule"]
    
    for split_mode in split_modes:
        print(f"\n{'='*80}")
        print(f"SPLIT MODE: {split_mode.upper()}")
        print(f"{'='*80}")
        
        # Load splits
        try:
            splits_full = load_splits(split_mode, base_dir="../dataset/splits")
            print(f"Loaded {len(splits_full)} folds for {split_mode} split")
        except Exception as e:
            print(f"⚠ Could not load splits for {split_mode}: {e}")
            continue
        
        # Filter splits to only include samples with pKd values
        splits_reg = []
        for train_idx, val_idx in splits_full:
            # Map original indices to filtered indices
            train_mask = valid_mask[train_idx]
            val_mask = valid_mask[val_idx]
            
            # Get filtered indices
            train_idx_filtered = np.where(valid_mask)[0]
            val_idx_filtered = np.where(valid_mask)[0]
            
            # Find which filtered indices correspond to original train/val
            train_idx_reg = np.array([i for i, orig_idx in enumerate(np.where(valid_mask)[0]) if orig_idx in train_idx])
            val_idx_reg = np.array([i for i, orig_idx in enumerate(np.where(valid_mask)[0]) if orig_idx in val_idx])
            
            if len(train_idx_reg) > 0 and len(val_idx_reg) > 0:
                splits_reg.append((train_idx_reg, val_idx_reg))
        
        print(f"  Filtered to {len(splits_reg)} folds with pKd values")
        
        if len(splits_reg) == 0:
            print(f"  ⚠ No valid folds for {split_mode}, skipping...")
            continue
        
        # Define regression model (same architecture as best classifier, but for regression)
        class TwoTowerCNNRegression(nn.Module):
            """TwoTower CNN model for regression."""
            def __init__(
                self,
                apt_dim,
                mol_dim,
                apt_top_type="cnn",
                mol_top_type="cnn",
                seq_len=32,
                hidden_dim=256,
                output_dim=128,
                fusion_hidden_dims=[512, 256, 128, 64],
                dropout=0.3,
                apt_top_kwargs=None,
                mol_top_kwargs=None
            ):
                super().__init__()
                
                self.seq_len = seq_len
                self.hidden_dim = hidden_dim
                
                # Project flat embeddings to sequences
                self.apt_seq_proj = nn.Sequential(
                    nn.Linear(apt_dim, seq_len * hidden_dim),
                    nn.LayerNorm(seq_len * hidden_dim)
                )
                self.mol_seq_proj = nn.Sequential(
                    nn.Linear(mol_dim, seq_len * hidden_dim),
                    nn.LayerNorm(seq_len * hidden_dim)
                )
                
                # Top heads
                top_classes = {
                    "identity": IdentityTop,
                    "cnn": CNNTop,
                    "lstm": LSTMTop,
                    "transformer": TransformerTop
                }
                
                # Filter kwargs
                def filter_kwargs(kwargs_dict):
                    if kwargs_dict is None:
                        return {}
                    return {k: v for k, v in kwargs_dict.items() 
                           if k not in ['seq_len', 'output_dim', 'fusion_hidden_dims', 'hidden_dim']}
                
                apt_kwargs = filter_kwargs(apt_top_kwargs)
                mol_kwargs = filter_kwargs(mol_top_kwargs)
                
                AptTopClass = top_classes[apt_top_type]
                MolTopClass = top_classes[mol_top_type]
                
                self.apt_top = AptTopClass(hidden_dim, output_dim, dropout=dropout, **apt_kwargs)
                self.mol_top = MolTopClass(hidden_dim, output_dim, dropout=dropout, **mol_kwargs)
                
                # Fusion layers (output single value for regression)
                fusion_dim = output_dim * 2
                fusion_layers = []
                prev_dim = fusion_dim
                for dim in fusion_hidden_dims:
                    fusion_layers.extend([
                        nn.Linear(prev_dim, dim),
                        nn.BatchNorm1d(dim),
                        nn.ReLU(),
                        nn.Dropout(dropout)
                    ])
                    prev_dim = dim
                fusion_layers.append(nn.Linear(prev_dim, 1))
                self.fusion = nn.Sequential(*fusion_layers)
            
            def forward(self, apt, mol):
                # Project to sequences
                apt_seq = self.apt_seq_proj(apt).view(-1, self.seq_len, self.hidden_dim)
                mol_seq = self.mol_seq_proj(mol).view(-1, self.seq_len, self.hidden_dim)
                
                # Apply top heads
                apt_emb = self.apt_top(apt_seq)
                mol_emb = self.mol_top(mol_seq)
                
                # Concatenate and fuse
                fused = torch.cat([apt_emb, mol_emb], dim=1)
                output = self.fusion(fused)
                return output.squeeze(-1)
        
        # Training function for regression
        def train_regression_model(
            model, train_loader, val_loader, optimizer, criterion, device,
            n_epochs=50, patience=15
        ):
            """Train regression model with early stopping."""
            best_val_loss = np.inf
            patience_counter = 0
            history = {
                'train_loss': [],
                'val_loss': [],
                'val_rmse': [],
                'val_mae': [],
                'val_r2': []
            }
            
            for epoch in range(n_epochs):
                # Training
                model.train()
                train_losses = []
                for batch in train_loader:
                    apt = batch['apt'].to(device)
                    mol = batch['mol'].to(device)
                    target = batch['label_reg'].to(device)
                    
                    optimizer.zero_grad()
                    pred = model(apt, mol)
                    loss = criterion(pred, target)
                    loss.backward()
                    optimizer.step()
                    
                    train_losses.append(loss.item())
                
                train_loss = np.mean(train_losses)
                
                # Validation
                model.eval()
                val_losses = []
                all_preds = []
                all_targets = []
                
                with torch.no_grad():
                    for batch in val_loader:
                        apt = batch['apt'].to(device)
                        mol = batch['mol'].to(device)
                        target = batch['label_reg'].to(device)
                        
                        pred = model(apt, mol)
                        loss = criterion(pred, target)
                        
                        val_losses.append(loss.item())
                        all_preds.extend(pred.cpu().numpy())
                        all_targets.extend(target.cpu().numpy())
                
                val_loss = np.mean(val_losses)
                all_preds = np.array(all_preds)
                all_targets = np.array(all_targets)
                
                # Compute metrics
                rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
                mae = mean_absolute_error(all_targets, all_preds)
                r2 = r2_score(all_targets, all_preds)
                
                # Update history
                history['train_loss'].append(train_loss)
                history['val_loss'].append(val_loss)
                history['val_rmse'].append(rmse)
                history['val_mae'].append(mae)
                history['val_r2'].append(r2)
                
                # Early stopping
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        break
            
            return history
        
        # Evaluate regression model
        def evaluate_regression(model, dataloader, device):
            """Evaluate regression model."""
            model.eval()
            all_preds = []
            all_targets = []
            
            with torch.no_grad():
                for batch in dataloader:
                    apt = batch['apt'].to(device)
                    mol = batch['mol'].to(device)
                    target = batch['label_reg'].cpu().numpy()
                    
                    pred = model(apt, mol).cpu().numpy()
                    
                    all_preds.extend(pred)
                    all_targets.extend(target)
            
            all_preds = np.array(all_preds)
            all_targets = np.array(all_targets)
            
            # Compute metrics
            rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
            mae = mean_absolute_error(all_targets, all_preds)
            r2 = r2_score(all_targets, all_preds)
            
            return {
                'RMSE': rmse,
                'MAE': mae,
                'R2': r2,
                'predictions': all_preds,
                'targets': all_targets
            }
        
        # Train on each fold
        fold_metrics = []
        fold_histories = []
        
        for fold_idx, (train_idx, val_idx) in enumerate(splits_reg):
            print(f"\n  Fold {fold_idx + 1}/{len(splits_reg)}...")
            
            # Split data
            apt_train, apt_val = apt_feat_reg[train_idx], apt_feat_reg[val_idx]
            mol_train, mol_val = mol_feat_reg[train_idx], mol_feat_reg[val_idx]
            y_train, y_val = y_reg_filtered[train_idx], y_reg_filtered[val_idx]
            
            # Print fold statistics
            print(f"    Train samples: {len(y_train)}, Val samples: {len(y_val)}")
            print(f"    Train pKd: {y_train.mean():.2f} ± {y_train.std():.2f}, Val pKd: {y_val.mean():.2f} ± {y_val.std():.2f}")
            print(f"    Training...", end=" ")
            
            # Normalize features
            scaler_apt = StandardScaler()
            scaler_mol = StandardScaler()
            apt_train = scaler_apt.fit_transform(apt_train)
            apt_val = scaler_apt.transform(apt_val)
            mol_train = scaler_mol.fit_transform(mol_train)
            mol_val = scaler_mol.transform(mol_val)
            
            # Create datasets (use dummy class labels)
            train_dataset = AptamerMoleculeDataset(
                apt_train, mol_train, np.zeros(len(y_train)), y_train
            )
            val_dataset = AptamerMoleculeDataset(
                apt_val, mol_val, np.zeros(len(y_val)), y_val
            )
            
            # Use drop_last=True for train to avoid BatchNorm error with batch_size=1
            train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)
            val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
            
            # Create model
            model = TwoTowerCNNRegression(
                apt_dim=apt_train.shape[1],
                mol_dim=mol_train.shape[1],
                apt_top_type="cnn",
                mol_top_type="cnn",
                seq_len=32,
                hidden_dim=256,
                output_dim=128,
                fusion_hidden_dims=[512, 256, 128, 64],
                dropout=0.3,
                apt_top_kwargs={"kernel_sizes": (3, 5, 7)},
                mol_top_kwargs={"kernel_sizes": (3, 5, 7)}
            ).to(device)
            
            # Optimizer and loss
            optimizer = Adam(model.parameters(), lr=1e-3)
            criterion = nn.MSELoss()
            
            # Train
            history = train_regression_model(
                model, train_loader, val_loader, optimizer, criterion, device,
                n_epochs=50, patience=15
            )
            fold_histories.append(history)
            
            # Final evaluation
            results = evaluate_regression(model, val_loader, device)
            fold_metrics.append(results)
            
            print(f"RMSE: {results['RMSE']:.3f}, MAE: {results['MAE']:.3f}, R²: {results['R2']:.3f}")
            
            # Clear memory
            del model
            clear_memory()
        
        # Aggregate metrics across folds
        rmse_values = [m['RMSE'] for m in fold_metrics]
        mae_values = [m['MAE'] for m in fold_metrics]
        r2_values = [m['R2'] for m in fold_metrics]
        
        result = {
            'split_mode': split_mode,
            'model': 'TwoTower_CNN_CNN',
            'apt_encoder': 'GENA',
            'mol_encoder': 'UniMol',
            'RMSE mean': np.mean(rmse_values),
            'RMSE std': np.std(rmse_values),
            'MAE mean': np.mean(mae_values),
            'MAE std': np.std(mae_values),
            'R2 mean': np.mean(r2_values),
            'R2 std': np.std(r2_values)
        }
        regression_results.append(result)
        
        print(f"\n  {'='*76}")
        print(f"  {split_mode.upper()} RESULTS:")
        print(f"  {'='*76}")
        print(f"  RMSE: {result['RMSE mean']:.3f} ± {result['RMSE std']:.3f}")
        print(f"  MAE:  {result['MAE mean']:.3f} ± {result['MAE std']:.3f}")
        print(f"  R²:   {result['R2 mean']:.3f} ± {result['R2 std']:.3f}")
    
    # Create results DataFrame
    if len(regression_results) > 0:
        regression_df = pd.DataFrame(regression_results)
        
        print(f"\n{'='*80}")
        print("REGRESSION MODEL RESULTS SUMMARY")
        print(f"{'='*80}")
        print(regression_df.to_string(index=False))
        
        # Save results
        regression_path = os.path.join(ARTIFACTS_DIR, "metrics", "regression_model_results.csv")
        regression_df.to_csv(regression_path, index=False)
        print(f"\n✓ Results saved to: {regression_path}")
    else:
        print("\n⚠ No results to save!")

print("\n" + "="*80)
print("REGRESSION MODEL EVALUATION COMPLETE")
print("="*80)


In [ ]:
# Fixed-path export check (reviewer reproducibility)
from pathlib import Path

required_outputs = [
    Path("../artifacts/dl/screening/comprehensive_screening_all_splits.csv"),
    Path("../artifacts/dl/screening/dl_leaderboard.csv"),
    Path("../artifacts/dl/ablations/ablation_study_gena_unimol.csv"),
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing expected DL outputs: {missing}")
print("Verified outputs:")
for path in required_outputs:
    print(f" - {path}")